# 1. 测试与部署

## stable diffusion v1-5
本地部署了半精度stable diffusion，简单研究diffuser的pipeline以及其中的调度方法和各种诸如alpha_prod等参数
简单复现了pipeline，同时保存了每个采样step上的图片，直接根据公式还原到X0之后的图片

In [ ]:
from diffusers import StableDiffusionPipeline
import torch

model_path = "D:\Code\python\ReinDiff\stable-diffusion-fp16"
pipe = StableDiffusionPipeline.from_pretrained(model_path, torch_dtype=torch.float16, variant='fp16').to('cuda')

In [ ]:
import torch
from transformers import AutoImageProcessor, Mask2FormerForUniversalSegmentation

device = 'cuda'
processor = AutoImageProcessor.from_pretrained('D:\models\mask2former')
model = Mask2FormerForUniversalSegmentation.from_pretrained('D:\models\mask2former').eval().to(device)

In [ ]:
# basic infos checking test
scheduler = pipe.scheduler

betas = torch.linspace(scheduler.beta_start, scheduler.beta_end, scheduler.num_train_timesteps)
alpha_t = 1 - betas
alpha_prod_t = scheduler.alphas_cumprod
scheduler.set_timesteps(num_inference_steps=50)
timesteps = scheduler.timesteps
print(timesteps, alpha_prod_t, alpha_t, sep='\n')

prompt = 'a photo of an astronaut riding a horse on Mars'
image = pipe(prompt).images[0]
image

In [ ]:
# show hidden step image test
import torch
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import gc

device = 'cuda'

prompt = "A photo of an astronaut riding a horse on Mars"
prompt_embed, empty_prompt_embed = pipe.encode_prompt(prompt, num_images_per_prompt=1, do_classifier_free_guidance=True, device=device)
embedding = torch.cat([empty_prompt_embed, prompt_embed]).to(torch.float16).to(device)  # donot use classifier free guidance

channels = pipe.unet.config.in_channels
size = 1
height = pipe.unet.config.sample_size
width = pipe.unet.config.sample_size
latents = torch.randn((size, channels, height, width), dtype=torch.float16, device=device)

num_inference_steps = 50
pipe.scheduler.set_timesteps(num_inference_steps=num_inference_steps, device=device)

generated_images = []
guidance_scale = 7.5
def reverse_origin(noise_pred, t, latents):
    alphas_cumprod = pipe.scheduler.alphas_cumprod
    alpha_prod_t = alphas_cumprod[t]
    beta_prod_t = 1 - alpha_prod_t
    x0 = (latents - (beta_prod_t ** 0.5)*noise_pred) / (alpha_prod_t ** 0.5)
    return x0

sample_gap = 1
for i, t in enumerate(pipe.scheduler.timesteps):
    print(f'handling {i}th loop')
    with torch.no_grad():
        latents_input = torch.cat([latents]*2)
        noise_pred = pipe.unet(latents_input, t, encoder_hidden_states=embedding).sample.to(device)  # sample outputs the (size, channels, height, width) data as current timestep image output
        noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
        noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)
        latents = pipe.scheduler.step(noise_pred, t, latents).prev_sample.to(device)  # calculate Xt-n based on predicted noise where n=1000/num_inference_steps, output Xt-1 is also shaped as (size, channels, height, width)
        latents_ori = reverse_origin(noise_pred, t, latents)
        if i % sample_gap == 0:
            image_t = pipe.decode_latents(latents)
            image_t_ori = pipe.decode_latents(latents_ori)
            image_t = Image.fromarray((image_t * 255).astype(np.uint8)[0])
            image_t_ori = Image.fromarray((image_t_ori * 255).astype(np.uint8)[0])
            image_t_ori.save(f"D:\Code\python\ReinDiff\images\dino\step_{i}.png")
            generated_images.append((image_t, image_t_ori))

gap = 10
n = len(generated_images) / gap
fig, axes = plt.subplots(n, 2, figsize=(2*n, 3*n))
for i, (img1, img2) in enumerate(generated_images):
    if i % gap == 0:
        axes[i, 0].imshow(img1)
        axes[i, 0].axis('off')
        axes[i, 1].imshow(img2)
        axes[i, 1].axis('off')
plt.tight_layout()
plt.show()

del prompt_embed, latents, noise_pred
torch.cuda.empty_cache()
gc.collect()


## DINO
使用一个仅100MB的小型ViT的最终的潜在层向量作为图片的向量，计算两个图片之间向量的余弦分数作为DINO来衡量二者的相似度。
流程和结果都是正确的，但是DINO需要一个标签图像作为对比。

In [ ]:
# calculate DINO score for all steps
from transformers import AutoModel, AutoImageProcessor
from PIL import Image
import numpy as np
import torch
device = 'cuda'
vit = AutoModel.from_pretrained('D:\models\dino-vits16').eval().to(device)
vit_processor = AutoImageProcessor.from_pretrained('D:\models\dino-vits16')

def extra_feature(img_path:str):
    img = Image.open(img_path).convert('RGB')
    inputs = vit_processor(images=img, return_tensors='pt')
    img_tensor = inputs['pixel_values'].to(device)
    with torch.no_grad():
        outputs = vit(img_tensor)
    cls_feature = outputs.last_hidden_state[:, 0, :]
    return cls_feature.cpu().numpy().squeeze()

def calc_dino(feat1, feat2):
    similarity = np.dot(feat1, feat2) / (np.linalg.norm(feat1) * np.linalg.norm(feat2))
    return float(similarity)

base = extra_feature('D:\Code\python\ReinDiff\images\dino\step_50.png')
similarities = []
for i in range(50):
    path = f'D:\Code\python\ReinDiff\images\dino\step_{i}.png'
    feat = extra_feature(path)
    similarities.append(calc_dino(feat, base))
print(similarities)


## mask2former
mask2former是一个语义分割模型
使用在ade20k训练集上训练的swin former模型，500MB左右的小模型做语义分割
失败之处在于stable diffusion生成的图片风格和特点等完全和原始数据集不同，因此这个语义分割网络是无法分辨的，遂把所有像素都当成背景了也就是全都是-1这个类别

In [ ]:
image = Image.open('D:\Code\python\ReinDiff\images\dino\step_50.png')
inputs = processor(images=image, return_tensors='pt').to(device)

with torch.no_grad():
    outputs = model(**inputs)

result = processor.post_process_instance_segmentation(outputs, target_sizes=[image.size[::-1]])[0]
predicted_instance_map = result["segmentation"]

print(outputs)

# 2. 思考与改进

## 强化学习的反思
事实证明，如果不去做supervised finetune，单纯的基于RM和语义分割去做强化学习不是很靠谱，原因是没有较好的反馈。
例如既然使用了looking Forward作为状态，那么DINO分数要和谁去对比呢？不能和最后一张图片对比，因为这样V就变成了和timestep相同意义的值了。
因此如果想要使用DINO和V就需要一个比较对象，这就变成了supervised fine-tune了，不像是强化学习了，因为强化学习往往需要RL。
所以如果需要RL，就需要RM，这时候looking Forward就需要RM来评分了，那么R又应该是什么呢？需要另外的一个奖励，就单纯是状态转移带来的奖励。

## 重新认识任务
语义分割引入diffusion，在最后几步中进行微调，attention map，准备coco数据集

In [ ]:
import requests
from PIL import Image
import io

image_url = 'http://images.cocodataset.org/train2017/000000391895.jpg'
response = requests.get(image_url)
image = Image.open(io.BytesIO(response.content))
image_resized = image.resize((512,512))
image_resized.save(f'D:\Code\python\ReinDiff\images\coco\download.png')

In [ ]:
from PIL import Image
import torch
import matplotlib.pyplot as plt
from torchvision import transforms
from torchinfo import summary

device = 'cuda'
image_path = f'D:\Code\python\ReinDiff\images\coco\download.png'
process_image = []
sample_gap = 1
# scheduler
scheduler = pipe.scheduler
alphas_cumprod = scheduler.alphas_cumprod
scheduler.set_timesteps(num_inference_steps=50)
timesteps = scheduler.timesteps
# vae
vae = pipe.vae
scaling_factor = 0.18215
# text_encoder
guidance_scale = 7.5
prompt = 'A man with a red helmet riding a motor bike on a dirt road on the countryside.'
prompt_embed, empty_prompt_embed = pipe.encode_prompt(prompt, num_images_per_prompt=1, do_classifier_free_guidance=True, device=device)
embedding = torch.cat([empty_prompt_embed, prompt_embed]).to(torch.float16).to(device)
# hook
attn_maps = {}
hooks = []
def hook(module, input, output):
    attn_maps[module] = output
for name, module in pipe.unet.named_modules():
    if 'attn' in name.lower():
        hooks.append(module.register_forward_hook(hook))


def calc_real_t(t):
    return int((t-1)*20 + 1)

def encode_image(path):
    image = Image.open(path).convert('RGB').resize((512,512))
    # vae希望图像像素值在[-1, 1]之间，不使用feature extractor否则会被缩放到224*224，手动实现归一化将图像的每个像素从[0,255]归一化到[-1,1]
    # image_tensor = pipe.feature_extractor(images=image, return_tensors='pt').pixel_values.to(device, dtype=torch.float16)
    trfms = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5])
    ])
    image_tensor = trfms(image).unsqueeze(0).to(dtype=torch.float16, device=device)
    with torch.no_grad():
        # vae训练得到的分布范围可能较大，而Unet训练时的latent的值的范围更小，因此需要scaling factor进行放缩
        latent = vae.encode(image_tensor).latent_dist.sample() * scaling_factor
    return latent

def add_noise(latent, t):
    epsilon = torch.randn_like(latent, dtype=torch.float16).to(device)
    alpha = alphas_cumprod[timesteps[-t]]
    noise_latent = (alpha**0.5) * latent + ((1 - alpha)**0.5) * epsilon
    return noise_latent

def predict(latent, t):
    with torch.no_grad():
        latent_input = torch.cat([latent] * 2)
        noise_pred = pipe.unet(latent_input, t, encoder_hidden_states=embedding).sample.to(device)
        noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
        noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)
        latent_pred = pipe.scheduler.step(noise_pred, t, latent).prev_sample.to(device)
    return latent_pred

def origin(noise_pred, t, latent):
    alpha_prod_t = alphas_cumprod[t]
    beta_prod_t = 1 - alpha_prod_t
    x0 = (latent - (beta_prod_t ** 0.5)*noise_pred) / (alpha_prod_t ** 0.5)
    return x0

def reverse(latent, t):
    for i, t in enumerate(timesteps[-t:]):
        print(f'handling {t}th loop')
        with torch.no_grad():
            latent_input = torch.cat([latent]*2)
            noise_pred = pipe.unet(latent_input, t, encoder_hidden_states=embedding).sample.to(device)  # sample outputs the (size, channels, height, width) data as current timestep image output
            noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
            noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)
            latent = pipe.scheduler.step(noise_pred, t, latent).prev_sample.to(device)  # calculate Xt-n based on predicted noise where n=1000/num_inference_steps, output Xt-1 is also shaped as (size, channels, height, width)
            latent_ori = origin(noise_pred, t, latent)
            if i % sample_gap == 0:
                image_t_ori = decode_latent(latent_ori)
                process_image.append(image_t_ori)
    return latent 

def decode_latent(latent):
    with torch.no_grad():
        decoded_image = vae.decode(latent / scaling_factor).sample
    # vae可能得到超出-1和1的维度，首先反向还原到0和1之间，使用clamp严格截断在0和1之间，然后使用permute调整元组顺序，原本是(C,H,W)现在为(H,W,C)
    decoded_image = (decoded_image / 2 + 0.5).clamp(0, 1)[0].permute(1, 2, 0).cpu().to(dtype=torch.float32).numpy()
    return decoded_image

def show(img_path, t):
    noise_image = decode_latent(add_noise(encode_image(img_path), t))
    result_image = decode_latent(reverse(add_noise(encode_image(img_path), t), t))
    real_timestep = calc_real_t(t)
    print(f'noise at {real_timestep} step')
    # visualize
    fig, axes = plt.subplots(1, 2, figsize=(10, 10))
    axes[0].imshow(noise_image)
    axes[0].axis('off')
    axes[0].set_title(f'noise image')
    axes[1].imshow(result_image)
    axes[1].axis('off')
    axes[1].set_title(f'result image')
    plt.imsave(f'D:\Code\python\ReinDiff\images\seg\demo.png', result_image)
    plt.tight_layout()
    plt.show()

def save():
    for t, image in enumerate(process_image):
        plt.imsave(f'D:\Code\python\ReinDiff\images\seg\step_{t}.png', image)

t = 10
show(image_path, t)
save()

# plt.imshow(decode_latent(predict(add_noise(encode_image(image_path), t), t)))
# plt.axis('off')
# plt.show()
# for h in hooks:
#     h.remove()
# for k, v in attn_maps.items():
#     print(k)
#     print(v)
# len(attn_maps)

In [ ]:
from PIL import Image
import torch
import numpy as np
import matplotlib.pyplot as plt
import json
from matplotlib.colors import ListedColormap


path_cal = f'D:\Code\python\ReinDiff\images\seg\demo.png'
path_real = f'D:\Code\python\ReinDiff\images\coco\download.png'
colors = [
    [(255, 0, 0), 'red'],
    [(0, 255, 0), 'green'],
    [(0, 0, 255), 'blue'],
    [(255, 255, 0), 'yellow'],
    [(255, 0, 255), 'pink'],
    [(0, 255, 255), 'purple']
]
with open('D:\models\mask2former\config.json', 'r') as f:
    id2label = json.load(f)['id2label']

def calc_real_t(t):
    return (t-1)*20 + 1

def segment_image(img_path):
    image = Image.open(img_path).convert('RGB')
    inputs = processor(images=image, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    result = processor.post_process_instance_segmentation(outputs, target_sizes=[image.size[::-1]])[0]
    segments_map = result['segmentation']
    segments_info = result['segments_info']
    return segments_map, segments_info

def visualize_image(segments_map, segments_info):
    color_desc = ''
    segments_image = np.zeros((segments_map.shape[0], segments_map.shape[1], 3))
    for i, label in enumerate(segments_info):
        color_desc += f"{colors[i][1]} represents {id2label[str(label['label_id'])]}\n"
        mask = segments_map == label['id']
        segments_image[mask] = colors[i][0]
    return segments_image, color_desc

def remap_image(segments_map, segments_info):
    # mask2former cannot distinguish the id of same label, maybe this time a_id=0&b_id=1, next time a_id=1&b_id=0, so try transfer to simple semantic segmentation
    labels = []
    # use for loop might cause issue like 0->1 then 1->2 thus all 0 and 1 become 2, but 0 suppose to be 1
    # for label in segments_info:
    #     mask = segments_map == label['id']
    #     segments_remap[mask] = label['label_id']
    #     labels.append(label['label_id'])
    # use np.take instead
    min_val = -1
    mapping_arr = np.zeros(len(segments_info)+1, dtype=int)
    mapping_arr[0] = -1
    for label in segments_info:
        mapping_arr[label['id']+1] = label['label_id']
        labels.append(label['label_id'])
    segments_remap = torch.tensor(np.take(mapping_arr, segments_map-min_val))
    return segments_remap, labels

def miou(segments_map_pred, segments_map_true, labels):
    ious = []
    for i in labels:
        tp = ((segments_map_pred == i) & (segments_map_true == i)).sum().item()
        fp = ((segments_map_pred == i) & (segments_map_true != i)).sum().item()
        fn = ((segments_map_pred != i) & (segments_map_true == i)).sum().item()
        if tp + fp + fn == 0:   iou = 1
        else: iou = tp / (tp + fp + fn)
        ious.append(iou)
    return sum(ious) / len(ious)
        
def show(img_path):
    # imshow the result of a segmented image, left is the original image, right is the segmented result
    segments_map, segments_info = segment_image(img_path)
    segments_image, color_map = visualize_image(segments_map, segments_info)
    image = Image.open(img_path).convert('RGB')
    print(color_map)
    fig, axes = plt.subplots(1, 2, figsize=(10, 10))
    axes[0].imshow(image)
    axes[0].axis('off')
    axes[1].imshow(segments_image)
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()

def visualize_remap_image(image1, image2):
    cmap = ListedColormap(["black", "white", "red", "blue", "green"])  # -1, 0, 1, 2, 3
    bounds = [-1.5, -0.5, 0.5, 1.5, 2.5, 3.5]
    norm = plt.cm.colors.BoundaryNorm(bounds, cmap.N)
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(image1, cmap=cmap, norm=norm)
    axes[0].set_title("pred image")
    axes[0].axis("off")
    axes[1].imshow(image2, cmap=cmap, norm=norm)
    axes[1].set_title("real image")
    axes[1].axis("off")
    plt.show()

def similarity(img_cal, img_real):
    segments_remap_true, labels = remap_image(*segment_image(img_real))
    segments_remap_pred, labels = remap_image(*segment_image(img_cal))  # use predicted segments_info as real info to punish extra class
    visualize_remap_image(segments_remap_pred, segments_remap_true)
    r = miou(segments_remap_pred, segments_remap_true, labels)
    return r

# show(path_cal)
# show(path_real)
similarities = []
t = 10
for i in range(t):
    if i % 3 == 0:
        path = f'D:\Code\python\ReinDiff\images\seg\step_{i}.png'
        show(path)
        print(calc_real_t(t-i))
        similarities.append(similarity(path, path_real))
similarities


## 实验基本已经全都完成，已经实现了语义分割、加噪还原、miou分数计算和可视化，接下来尝试进行微调
首先简单尝试下载少量数据，然后进行微调跑通流程

In [ ]:
# download dataset
from datasets import load_dataset
import os
import io
import requests
import traceback
from PIL import Image

dataset = load_dataset('parquet', data_files=f'D:\Code\python\ReinDiff\images\coco\data.parquet')
save_folder = f'D:\Code\python\ReinDiff\images\coco\data_train'
for i in range(10):
    url = dataset['train'][i]['coco_url']
    file_name = str(dataset['train'][i]['file_name']).replace('/', '_')
    save_path = os.path.join(save_folder, file_name)
    try:
        response = requests.get(url, timeout=5)
        image = Image.open(io.BytesIO(response.content)).convert('RGB')
        image.save(save_path)
    except:
        traceback.print_exc()


In [ ]:
import torch
from torch.optim import AdamW
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset
from torch.utils.data import DataLoader
from torchvision import transforms
from PIL import Image

# define configuration
lora_rank = 4
lora_alpha = 32
lora_dropout = 0.1
batch_size = 2
learning_rate = 1e-5
num_epoches = 1

scaling_factor = 0.18215
guidance_scale = 7.5
num_inference_steps = 50


# acquire tools
unet = pipe.unet
prompt = 'A man with a red helmet riding a motor bike on a dirt road on the countryside.'
device = 'cuda'
vae = pipe.vae
scheduler = pipe.scheduler
alphas_cumprod = scheduler.alphas_cumprod

# basic preparations
prompt_embed, empty_prompt_embed = pipe.encode_prompt(prompt, num_images_per_prompt=1, do_classifier_free_guidance=True, device=device)
embedding = torch.cat([empty_prompt_embed, prompt_embed]).to(torch.float16).to(device)

scheduler.set_timesteps(num_inference_steps=num_inference_steps)
timesteps = scheduler.timesteps

with open('D:\models\mask2former\config.json', 'r') as f:
    id2label = json.load(f)['id2label']


# config lora
lora_config = LoraConfig(
    r=lora_rank, 
    lora_alpha=lora_alpha,
    target_modules=['to_q', 'to_v'],
    lora_dropout=lora_dropout,
    task_type=TaskType.FEATURE_EXTRACTION
)
unet = get_peft_model(unet, lora_config)


# image preprocess
transform = transforms.Compose([
    transforms.Resize((512, 512)),  # resize the target image to 512*512
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])
def preprocess(batch):
    images = [transform(item['image']) for item in batch]
    return torch.stack(images)

# load dataset
save_folder = f'D:\Code\python\ReinDiff\images\coco\data_train'
dataset = load_dataset('imagefolder', data_dir=f'D:\Code\python\ReinDiff\images\coco\data_train')
dataloader = DataLoader(dataset['train'], batch_size=batch_size, shuffle=True, collate_fn=preprocess)


# config training
optimizer = AdamW(unet.parameters(), lr=learning_rate)
unet.train()


# library functions
def encode_image(image:torch.Tensor):
    with torch.no_grad():
        latent = vae.encode(image).latent_dist.sample() * scaling_factor
    return latent

def decode_latent(latent):
    with torch.no_grad():
        decoded_image = vae.decode(latent / scaling_factor).sample
        decoded_image = (decoded_image / 2 + 0.5).clamp(0, 1)[0].permute(1, 2, 0).cpu().to(dtype=torch.float32).numpy()
    return decoded_image

def add_nosie(latent, t):
    epsilon = torch.randn_like(latent, dtype=torch.float16).to(device)
    alpha = alphas_cumprod[timesteps[-t]]
    noise_latent = (alpha**0.5) * latent + ((1 - alpha)**0.5) * epsilon
    return noise_latent

def predict(latent, t):
    with torch.no_grad():
        latent_cfg = torch.cat([latent] * 2)
        noise_pred = unet(latent_cfg, t, encoder_hidden_states=embedding).sample.to(device)
        noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
        noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)
        latent_pred = pipe.scheduler.step(noise_pred, t, latent).prev_sample.to(device)
    return noise_pred, latent_pred

def origin(noise_pred, t, latent):
    with torch.no_grad():
        alpha_prod_t = alphas_cumprod[t]
        beta_prod_t = 1 - alpha_prod_t
        x0 = (latent - (beta_prod_t ** 0.5)*noise_pred) / (alpha_prod_t ** 0.5)
    return x0

def segment_image(image):
    inputs = processor(images=image, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    result = processor.post_process_instance_segmentation(outputs, target_sizes=[image.size[::-1]])[0]
    segments_map = result['segmentation']
    segments_info = result['segments_info']
    return segments_map, segments_info

def remap_image(segments_map, segments_info):
    labels = []
    min_val = -1
    mapping_arr = np.zeros(len(segments_info)+1, dtype=int)
    mapping_arr[0] = -1
    for label in segments_info:
        mapping_arr[label['id']+1] = label['label_id']
        labels.append(label['label_id'])
    segments_remap = torch.tensor(np.take(mapping_arr, segments_map-min_val))
    return segments_remap, labels

def miou(segments_map_pred, segments_map_true, labels):
    ious = []
    for i in labels:  # labels suppose to be segments_map_pred's label which can distinguish extra class
        tp = ((segments_map_pred == i) & (segments_map_true == i)).sum().item()
        fp = ((segments_map_pred == i) & (segments_map_true != i)).sum().item()
        fn = ((segments_map_pred != i) & (segments_map_true == i)).sum().item()
        if tp + fp + fn == 0:   iou = 1
        else: iou = tp / (tp + fp + fn)
        ious.append(iou)
    return sum(ious) / len(ious)


# start training
for epoch in range(num_epoches):
    print(f'training epoch {epoch}')
    for batch_data in dataloader:
        batch_data = batch_data.to(device, dtype=torch.float16)




In [ ]:

# def similar_tokens(self, prompt:str):
    #     tokens = self.get_tokens(prompt)
    #     startindex, endindex = 1, len(tokens) - 1  # remove '<|startoftext|>' and '<|endoftext|>'
    #     token_embeddings = self.wordsim.batch_embedding(tokens[startindex:endindex])
    #     type_embeddings = self.wordsim.batch_embedding(self.types)
    #     similar = []
    #     threshold = 0.7
    #     for i, token_embedding in enumerate(token_embeddings):
    #         for j, type_embedding in enumerate(type_embeddings):
    #             if self.wordsim.similarity(type_embedding, token_embedding) > threshold:
    #                 similar.append((i+startindex, str(j)))
    #                 break
    #     return similar


def frozen():
    for name, param in unet.named_parameters():
        if 'lora_' not in name:
            param.requires_grad = False


def similar_tokens(self, prompt:str):
    tokens = self.get_tokens(prompt)
    startindex, endindex = 1, len(tokens) - 1  # remove '<|startoftext|>' and '<|endoftext|>'
    doc_tokens = [self.spacy(token)[0] for token in tokens[startindex:endindex]]
    doc_types = [self.spacy(type)[0] for type in self.types]
    threshold = 0.55
    results = []
    for i, token in enumerate(doc_tokens):
        similars = []
        for j, type in enumerate(doc_types):
            score = self.spacy.similarity(token, type)
            if self.spacy.is_noun(token) and score > threshold:
                similars.append((score, type))
        if len(similars) == 0: continue
        similars.sort(key=lambda x:x[0], reverse=True)
        _, type = similars[0]
        results.append((i+startindex, type))
    return results

In [ ]:
# from sdlib.lib import lib
# # from sdlib.vis import vis
# from sdlib.config import *
# lib.download_data(parquet_path, train_path, annotation_path, 0, 10)

# import json
# with open('D:\Code\python\ReinDiff\images\coco\instances_train2017.json', 'r') as f:
#     s = json.load(f)
# # print(s.keys())
# # print(s['info'], s['licenses'], s['categories'])
# print(s['images'][:5])
# print(s['annotations'][:5])


# r = datasetlib.seg_mask(391895)
# vislib.visualize_image(r)
# results = datasetlib.sep_mask(r)
# for id, img in results:
#     print(id)
#     vislib.visualize_image(img)


# t_ori = 5
# path = 'D:\\Code\\python\\ReinDiff\\images\\coco\\data_train\\train2017_000000391895.jpg'
# prompt = "A man with a red helmet riding motorbike on a dirt road."

# img = lib.read_image(path, 'Image')
# prompt, info = lib.seg_sep_prompt_image(img)
# print(prompt)
# for idx, label, obj, map in info:
#     print(f'{idx} refers to {obj}')
#     vis.visualize_image(map)

# prompt = "A photo of person motorbike"

# # img = lib.read_image(path, 'Image')
# # prompt, map = lib.generate_prompt(img)
# # print(prompt)
# # for index, label in map:
# #     print(index, label)

# # img = lib.read_image(path, 'Image')
# # r = lib.sepnseg_image(img)
# # for img, label in r:
# #     img = lib.downsample_image(img)
# #     vis.visualize_image(img)

# lib.attn_scores_hook()
# t = lib.calc_t(t_ori)
# num_tokens = lib.cnt_tokens(prompt)
# image = lib.read_image(path, 'Tensor', convert=True)

# noise_latent = lib.add_noise(lib.encode_image(image), t)
# reverse_latent = lib.origin(noise_latent, t, lib.encode_prompt(prompt))
# reverse_image = lib.decode_latent(reverse_latent)

# attn_scores = lib.attn_scores_numpy
# for i, attn_score in enumerate(attn_scores):
#     print(f'{i}: {attn_score.shape}')

# target = attn_scores[6]
# vis.visualize_attn_score(target, num_tokens=num_tokens)


In [ ]:
import torch
import numpy as np
from scipy.ndimage import zoom
from typing import Literal
from diffusers.models.attention_processor import Attention, AttnProcessor
from typing import Optional

def normalize_image_numpy(images: np.ndarray, norm: Literal['min_max', 'z_score']) -> np.ndarray:
    # images: [t, n, n]
    if norm == 'min_max':
        img_min = images.min(axis=(1, 2), keepdims=True)
        img_max = images.max(axis=(1, 2), keepdims=True)
        diff = img_max - img_min
        diff[diff < 1e-5] = 1e-5
        return (images - img_min) / diff

    elif norm == 'z_score':
        mean = images.mean(axis=(1, 2), keepdims=True)
        std = images.std(axis=(1, 2), keepdims=True)
        std[std < 1e-5] = 1e-5
        z = (images - mean) / std
        return (np.tanh(z) + 1) / 2

def resample_image_numpy(images: np.ndarray, size: tuple = (64, 64)) -> np.ndarray:
    # images: [t, n, n]
    t, h, w = images.shape
    zoom_factors = (1.0, size[0] / h, size[1] / w)
    resampled = zoom(images, zoom=zoom_factors, order=0)
    return resampled

def normalize_image_tensor(image:torch.Tensor) -> torch.Tensor:
    # image is attention scores like (77, n, n)
    num_tokens = image.shape[0]
    min_vals = image.view(num_tokens, -1).min(dim=1)[0].view(num_tokens, 1, 1)
    max_vals = image.view(num_tokens, -1).max(dim=1)[0].view(num_tokens, 1, 1)
    return (image - min_vals) / (max_vals - min_vals + 1e-8)

def batch_resample_image_tensor(image:torch.Tensor, size:tuple=(64, 64)) -> torch.Tensor:
    # image shape like (t, n, n)
    h, w = image.shape
    if (h, w) == size:
        return image
    image_reshaped = image.unsqueeze(1)  # (t, 1, n, n)
    image_upsampled = torch.nn.functional.interpolate(image_reshaped, size=size, mode='bilinear', align_corners=False)
    image = image_upsampled.squeeze(1)
    return image

def resample_image_tensor(image:torch.Tensor, size:tuple=(64, 64)) -> torch.Tensor:
    # image shape like (n, n)
    h, w = image.shape
    if (h, w) == size:
        return image
    mask_reshaped = image.unsqueeze(0).unsqueeze(0)
    mask_downsampled = torch.nn.functional.interpolate(mask_reshaped, size=size, mode='bilinear', align_corners=False)
    image = mask_downsampled.squeeze(0).squeeze(0)
    return image

class CustomAttnProcessor(AttnProcessor):
    def __init__(self):
        super().__init__()
        self.attn_score = None
    
    def calc_attn_logits(
            self, query:torch.Tensor, key:torch.Tensor, attention_mask:torch.Tensor = None, head_dim:int=64
        ) -> torch.Tensor:
        dtype = query.dtype
        dim_head = head_dim
        if attention_mask is None:
            baddbmm_input = torch.empty(
                query.shape[0], query.shape[1], key.shape[1], dtype=query.dtype, device=query.device
            )
            beta = 0
        else:
            baddbmm_input = attention_mask
            beta = 1
        scale = dim_head**-0.5
        attention_scores = torch.baddbmm(
            baddbmm_input,
            query,
            key.transpose(-1, -2),
            beta=beta,
            alpha=scale,
        )
        del baddbmm_input
        attention_scores = attention_scores.to(dtype)
        return attention_scores

    def __call__(
            self,
            attn:Attention,
            hidden_states: torch.Tensor,
            encoder_hidden_states: Optional[torch.Tensor] = None,
            attention_mask: Optional[torch.Tensor] = None,
            temb: Optional[torch.Tensor] = None,
            *args, 
            **kwargs
        )->torch.Tensor:
        residual = hidden_states
        if attn.spatial_norm is not None:
            hidden_states = attn.spatial_norm(hidden_states, temb)

        input_ndim = hidden_states.ndim

        if input_ndim == 4:
            batch_size, channel, height, width = hidden_states.shape
            hidden_states = hidden_states.view(batch_size, channel, height * width).transpose(1, 2)

        batch_size, sequence_length, _ = (
            hidden_states.shape if encoder_hidden_states is None else encoder_hidden_states.shape
        )

        if attention_mask is not None:
            attention_mask = attn.prepare_attention_mask(attention_mask, sequence_length, batch_size)
            # scaled_dot_product_attention expects attention_mask shape to be
            # (batch, heads, source_length, target_length)
            attention_mask = attention_mask.view(batch_size, attn.heads, -1, attention_mask.shape[-1])

        if attn.group_norm is not None:
            hidden_states = attn.group_norm(hidden_states.transpose(1, 2)).transpose(1, 2)

        query = attn.to_q(hidden_states)

        if encoder_hidden_states is None:
            encoder_hidden_states = hidden_states
        elif attn.norm_cross:
            encoder_hidden_states = attn.norm_encoder_hidden_states(encoder_hidden_states)

        key = attn.to_k(encoder_hidden_states)
        value = attn.to_v(encoder_hidden_states)


        inner_dim = key.shape[-1]
        head_dim = inner_dim // attn.heads

        '''
        hidden state有两个维度分别是Classifier free的和真实的,因此to_q to_k to_v之后也是两个维度,只有第2个维度是有用的,添加一行代码：
        此外上面的to_q to_k to_v都是已经注入了lora的结果,因此计算图中可以囊括lora模块并且拿到的也是lora之后的attention score
        '''
        self.attn_score = self.calc_attn_logits(query, key, attention_mask, head_dim)[1]

        query = query.view(batch_size, -1, attn.heads, head_dim).transpose(1, 2)

        key = key.view(batch_size, -1, attn.heads, head_dim).transpose(1, 2)
        value = value.view(batch_size, -1, attn.heads, head_dim).transpose(1, 2)

        if attn.norm_q is not None:
            query = attn.norm_q(query)
        if attn.norm_k is not None:
            key = attn.norm_k(key)

        # the output of sdp = (batch, num_heads, seq_len, head_dim)
        # TODO: add support for attn.scale when we move to Torch 2.1
        hidden_states = torch.nn.functional.scaled_dot_product_attention(
            query, key, value, attn_mask=attention_mask, dropout_p=0.0, is_causal=False
        )

        hidden_states = hidden_states.transpose(1, 2).reshape(batch_size, -1, attn.heads * head_dim)
        hidden_states = hidden_states.to(query.dtype)

        # linear proj
        hidden_states = attn.to_out[0](hidden_states)
        # dropout
        hidden_states = attn.to_out[1](hidden_states)

        if input_ndim == 4:
            hidden_states = hidden_states.transpose(-1, -2).reshape(batch_size, channel, height, width)

        if attn.residual_connection:
            hidden_states = hidden_states + residual

        hidden_states = hidden_states / attn.rescale_output_factor

        return hidden_states



In [ ]:
# difflib
from model.stable_diffusion import stable_diffusion
import torch
from torchvision import transforms
from PIL import Image
import numpy as np
from typing import Literal
from typing import Literal
from diffusers.models.attention_processor import Attention, AttnProcessor
from diffusers.models.unets.unet_2d_condition import UNet2DConditionModel
from typing import Optional


pipe = stable_diffusion.pipe
tokenizer = stable_diffusion.pipe.tokenizer
unet = stable_diffusion.pipe.unet
scheduler = stable_diffusion.pipe.scheduler
vae = stable_diffusion.pipe.vae
# parameters
scaling_factor = 0.18215
guidance_scale = 7.5
num_inference_steps = 50
transformer = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])
dataloader = None
scheduler.set_timesteps(num_inference_steps=num_inference_steps)
timesteps = scheduler.timesteps
alphas_cumprod = scheduler.alphas_cumprod
device = 'cuda'
dtype = torch.float16

def calc_t(t:int):
        return timesteps[-t]

def read_image(path:str, type:Literal['Tensor', 'Image', 'Numpy'], convert:bool=False)->torch.Tensor|Image.Image|np.ndarray:
    image = Image.open(path).resize((512, 512), resample=Image.BILINEAR)
    # convert
    if convert:
        image = image.convert('RGB')
    # type
    if type == 'Tensor':
        image = transformer(image).unsqueeze(0).to(device=device, dtype=dtype)
    elif type == 'Numpy':
        image = np.array(image)
    return image

def cnt_tokens(prompt:str):
    text_inputs = tokenizer(prompt, padding=False, truncation=True, return_tensors='pt')
    text_input_ids = text_inputs.input_ids
    return text_input_ids.shape[1]

def get_tokens(prompt:str):
    text_inputs = tokenizer(prompt, padding=False, truncation=True, return_tensors=None)
    text_inputs_ids = text_inputs['input_ids']
    tokens = tokenizer.convert_ids_to_tokens(text_inputs_ids)
    tokens = [str(token).rstrip('</w>') for token in tokens]
    return tokens

def encode_prompt(prompt:str):
    prompt_embed, empty_prompt_embed = pipe.encode_prompt(prompt, num_images_per_prompt=1, do_classifier_free_guidance=True, device=device)
    return torch.cat([empty_prompt_embed, prompt_embed]).to(device=device, dtype=dtype)

def encode_image(image:torch.Tensor):
    # vae encoded value on every pixel is a little high, thus multiply a scaling_factor to fit unet input
    latent = vae.encode(image).latent_dist.sample() * scaling_factor
    return latent

def decode_latent(latent):
    image = vae.decode(latent / scaling_factor).sample
    image = (image / 2 + 0.5).clamp(0, 1)[0].permute(1, 2, 0).to(dtype=torch.float32).detach().cpu().numpy()
    return image

def add_noise(latent, t):
    epsilon = torch.randn_like(latent, dtype=dtype).to(device)
    alpha = alphas_cumprod[t]
    noise_latent = (alpha**0.5) * latent + ((1 - alpha)**0.5) * epsilon
    return epsilon, noise_latent

def predict(unet:UNet2DConditionModel, latent, t, embedding):
    latent_cfg = torch.cat([latent] * 2)
    noise_pred = unet(latent_cfg, t, encoder_hidden_states=embedding).sample.to(device)
    noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
    noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)
    return noise_pred

def step(latent, t, noise_pred):
    latent_pred = scheduler.step(noise_pred, t, latent).prev_sample.to(device)
    return latent_pred

def origin(latent, t, noise_pred):
    alpha_prod_t = alphas_cumprod[t]
    beta_prod_t = 1 - alpha_prod_t
    x0_latent = (latent - (beta_prod_t ** 0.5)*noise_pred) / (alpha_prod_t ** 0.5)
    return x0_latent

def resample(image:torch.Tensor, size:tuple=(64, 64)) -> torch.Tensor:
    # image shape like (n, n)
    h, w = image.shape
    if (h, w) == size:
        return image
    image_reshaped = image.unsqueeze(0).unsqueeze(0)
    image_downsampled = torch.nn.functional.interpolate(image_reshaped, size=size, mode='bilinear', align_corners=False)
    image = image_downsampled.squeeze(0).squeeze(0)
    return image

def pipeline(path:str, unet:UNet2DConditionModel, t_ddim:int, prompt:str):
    t = calc_t(t_ddim)
    image = read_image(path, 'Tensor')
    embedding = encode_prompt(prompt)
    latent = encode_image(image)
    _, noise_latent = add_noise(latent, t)
    noise_pred = predict(unet, noise_latent, t, embedding)
    x0_latent = origin(noise_latent, t, noise_pred)
    return decode_latent(x0_latent)


class CustomAttnProcessor(AttnProcessor):
    def __init__(self):
        super().__init__()
        self.attn_score = None
    
    def calc_attn_logits(
            self, query:torch.Tensor, key:torch.Tensor, attention_mask:torch.Tensor = None, head_dim:int=64
        ) -> torch.Tensor:
        dtype = query.dtype
        dim_head = head_dim
        if attention_mask is None:
            baddbmm_input = torch.empty(
                query.shape[0], query.shape[1], key.shape[1], dtype=query.dtype, device=query.device
            )
            beta = 0
        else:
            baddbmm_input = attention_mask
            beta = 1
        scale = dim_head**-0.5
        attention_scores = torch.baddbmm(
            baddbmm_input,
            query,
            key.transpose(-1, -2),
            beta=beta,
            alpha=scale,
        )
        del baddbmm_input
        attention_scores = attention_scores.to(dtype)
        return attention_scores

    def __call__(
            self,
            attn:Attention,
            hidden_states: torch.Tensor,
            encoder_hidden_states: Optional[torch.Tensor] = None,
            attention_mask: Optional[torch.Tensor] = None,
            temb: Optional[torch.Tensor] = None,
            *args, 
            **kwargs
        )->torch.Tensor:
        residual = hidden_states
        if attn.spatial_norm is not None:
            hidden_states = attn.spatial_norm(hidden_states, temb)

        input_ndim = hidden_states.ndim

        if input_ndim == 4:
            batch_size, channel, height, width = hidden_states.shape
            hidden_states = hidden_states.view(batch_size, channel, height * width).transpose(1, 2)

        batch_size, sequence_length, _ = (
            hidden_states.shape if encoder_hidden_states is None else encoder_hidden_states.shape
        )

        if attention_mask is not None:
            attention_mask = attn.prepare_attention_mask(attention_mask, sequence_length, batch_size)
            # scaled_dot_product_attention expects attention_mask shape to be
            # (batch, heads, source_length, target_length)
            attention_mask = attention_mask.view(batch_size, attn.heads, -1, attention_mask.shape[-1])

        if attn.group_norm is not None:
            hidden_states = attn.group_norm(hidden_states.transpose(1, 2)).transpose(1, 2)

        query = attn.to_q(hidden_states)

        if encoder_hidden_states is None:
            encoder_hidden_states = hidden_states
        elif attn.norm_cross:
            encoder_hidden_states = attn.norm_encoder_hidden_states(encoder_hidden_states)

        key = attn.to_k(encoder_hidden_states)
        value = attn.to_v(encoder_hidden_states)


        inner_dim = key.shape[-1]
        head_dim = inner_dim // attn.heads

        '''
        hidden state有两个维度分别是Classifier free的和真实的,因此to_q to_k to_v之后也是两个维度,只有第2个维度是有用的,添加一行代码：
        此外上面的to_q to_k to_v都是已经注入了lora的结果,因此计算图中可以囊括lora模块并且拿到的也是lora之后的attention score
        '''
        self.attn_score = self.calc_attn_logits(query, key, attention_mask, head_dim)[1]

        query = query.view(batch_size, -1, attn.heads, head_dim).transpose(1, 2)

        key = key.view(batch_size, -1, attn.heads, head_dim).transpose(1, 2)
        value = value.view(batch_size, -1, attn.heads, head_dim).transpose(1, 2)

        if attn.norm_q is not None:
            query = attn.norm_q(query)
        if attn.norm_k is not None:
            key = attn.norm_k(key)

        # the output of sdp = (batch, num_heads, seq_len, head_dim)
        # TODO: add support for attn.scale when we move to Torch 2.1
        hidden_states = torch.nn.functional.scaled_dot_product_attention(
            query, key, value, attn_mask=attention_mask, dropout_p=0.0, is_causal=False
        )

        hidden_states = hidden_states.transpose(1, 2).reshape(batch_size, -1, attn.heads * head_dim)
        hidden_states = hidden_states.to(query.dtype)

        # linear proj
        hidden_states = attn.to_out[0](hidden_states)
        # dropout
        hidden_states = attn.to_out[1](hidden_states)

        if input_ndim == 4:
            hidden_states = hidden_states.transpose(-1, -2).reshape(batch_size, channel, height, width)

        if attn.residual_connection:
            hidden_states = hidden_states + residual

        hidden_states = hidden_states / attn.rescale_output_factor

        return hidden_states

In [ ]:
import torch
from torchvision import transforms
from PIL import Image
import numpy as np
from typing import Literal
from diffusers.models.unets.unet_2d_condition import UNet2DConditionModel
from transformers.models.clip.tokenization_clip import CLIPTokenizer
from diffusers import StableDiffusionPipeline
from diffusers.models.autoencoders.autoencoder_kl import AutoencoderKL
from diffusers.schedulers.scheduling_pndm import PNDMScheduler
from typing import Literal
from diffusers.models.attention_processor import Attention, AttnProcessor
from typing import Optional


scaling_factor = 0.18215
guidance_scale = 7.5
transformer = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])
device = 'cuda'
dtype = torch.float16


def calc_t(scheduler:PNDMScheduler, t:int):
        timesteps = scheduler.timesteps
        return timesteps[-t]

def read_image(path:str, type:Literal['Tensor', 'Image', 'Numpy'], convert:bool=False)->torch.Tensor|Image.Image|np.ndarray:
    image = Image.open(path).resize((512, 512), resample=Image.BILINEAR)
    # convert
    if convert:
        image = image.convert('RGB')
    # type
    if type == 'Tensor':
        image = transformer(image).unsqueeze(0).to(device=device, dtype=dtype)
    elif type == 'Numpy':
        image = np.array(image)
    return image

def cnt_tokens(tokenizer:CLIPTokenizer, prompt:str):
    text_inputs = tokenizer(prompt, padding=False, truncation=True, return_tensors='pt')
    text_input_ids = text_inputs.input_ids
    return text_input_ids.shape[1]

def get_tokens(tokenizer:CLIPTokenizer, prompt:str):
    text_inputs = tokenizer(prompt, padding=False, truncation=True, return_tensors=None)
    text_inputs_ids = text_inputs['input_ids']
    tokens = tokenizer.convert_ids_to_tokens(text_inputs_ids)
    tokens = [str(token).rstrip('</w>') for token in tokens]
    return tokens

def encode_prompt(pipe:StableDiffusionPipeline, prompt:str):
    prompt_embed, empty_prompt_embed = pipe.encode_prompt(prompt, num_images_per_prompt=1, do_classifier_free_guidance=True, device=device)
    return torch.cat([empty_prompt_embed, prompt_embed]).to(device=device, dtype=dtype)

def encode_image(vae:AutoencoderKL, image:torch.Tensor):
    # vae encoded value on every pixel is a little high, thus multiply a scaling_factor to fit unet input
    latent = vae.encode(image).latent_dist.sample() * scaling_factor
    return latent

def decode_latent(vae:AutoencoderKL, latent):
    image = vae.decode(latent / scaling_factor).sample
    image = (image / 2 + 0.5).clamp(0, 1)[0].permute(1, 2, 0).to(dtype=torch.float32).detach().cpu().numpy()
    return image

def add_noise(scheduler:PNDMScheduler, latent, t):
    epsilon = torch.randn_like(latent, dtype=dtype).to(device)
    alpha = scheduler.alphas_cumprod[t]
    noise_latent = (alpha**0.5) * latent + ((1 - alpha)**0.5) * epsilon
    return epsilon, noise_latent

def predict(unet:UNet2DConditionModel, latent, t, embedding):
    latent_cfg = torch.cat([latent] * 2)
    noise_pred = unet(latent_cfg, t, encoder_hidden_states=embedding).sample.to(device)
    noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
    noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)
    return noise_pred

def step(scheduler:PNDMScheduler, latent, t, noise_pred):
    latent_pred = scheduler.step(noise_pred, t, latent).prev_sample.to(device)
    return latent_pred

def origin(scheduler:PNDMScheduler, latent, t, noise_pred):
    alpha_prod_t = scheduler.alphas_cumprod[t]
    beta_prod_t = 1 - alpha_prod_t
    x0_latent = (latent - (beta_prod_t ** 0.5)*noise_pred) / (alpha_prod_t ** 0.5)
    return x0_latent

def resample(image:torch.Tensor, size:tuple=(64, 64)) -> torch.Tensor:
    # image shape like (n, n)
    h, w = image.shape
    if (h, w) == size:
        return image
    image_reshaped = image.unsqueeze(0).unsqueeze(0)
    image_downsampled = torch.nn.functional.interpolate(image_reshaped, size=size, mode='bilinear', align_corners=False)
    image = image_downsampled.squeeze(0).squeeze(0)
    return image

class CustomAttnProcessor(AttnProcessor):
    def __init__(self):
        super().__init__()
        self.attn_score = None
    
    def calc_attn_logits(
            self, query:torch.Tensor, key:torch.Tensor, attention_mask:torch.Tensor = None, head_dim:int=64
        ) -> torch.Tensor:
        dtype = query.dtype
        dim_head = head_dim
        if attention_mask is None:
            baddbmm_input = torch.empty(
                query.shape[0], query.shape[1], key.shape[1], dtype=query.dtype, device=query.device
            )
            beta = 0
        else:
            baddbmm_input = attention_mask
            beta = 1
        scale = dim_head**-0.5
        attention_scores = torch.baddbmm(
            baddbmm_input,
            query,
            key.transpose(-1, -2),
            beta=beta,
            alpha=scale,
        )
        del baddbmm_input
        attention_scores = attention_scores.to(dtype)
        return attention_scores

    def __call__(
            self,
            attn:Attention,
            hidden_states: torch.Tensor,
            encoder_hidden_states: Optional[torch.Tensor] = None,
            attention_mask: Optional[torch.Tensor] = None,
            temb: Optional[torch.Tensor] = None,
            *args, 
            **kwargs
        )->torch.Tensor:
        residual = hidden_states
        if attn.spatial_norm is not None:
            hidden_states = attn.spatial_norm(hidden_states, temb)

        input_ndim = hidden_states.ndim

        if input_ndim == 4:
            batch_size, channel, height, width = hidden_states.shape
            hidden_states = hidden_states.view(batch_size, channel, height * width).transpose(1, 2)

        batch_size, sequence_length, _ = (
            hidden_states.shape if encoder_hidden_states is None else encoder_hidden_states.shape
        )

        if attention_mask is not None:
            attention_mask = attn.prepare_attention_mask(attention_mask, sequence_length, batch_size)
            # scaled_dot_product_attention expects attention_mask shape to be
            # (batch, heads, source_length, target_length)
            attention_mask = attention_mask.view(batch_size, attn.heads, -1, attention_mask.shape[-1])

        if attn.group_norm is not None:
            hidden_states = attn.group_norm(hidden_states.transpose(1, 2)).transpose(1, 2)

        query = attn.to_q(hidden_states)

        if encoder_hidden_states is None:
            encoder_hidden_states = hidden_states
        elif attn.norm_cross:
            encoder_hidden_states = attn.norm_encoder_hidden_states(encoder_hidden_states)

        key = attn.to_k(encoder_hidden_states)
        value = attn.to_v(encoder_hidden_states)


        inner_dim = key.shape[-1]
        head_dim = inner_dim // attn.heads

        '''
        hidden state有两个维度分别是Classifier free的和真实的,因此to_q to_k to_v之后也是两个维度,只有第2个维度是有用的,添加一行代码：
        此外上面的to_q to_k to_v都是已经注入了lora的结果,因此计算图中可以囊括lora模块并且拿到的也是lora之后的attention score
        '''
        self.attn_score = self.calc_attn_logits(query, key, attention_mask, head_dim)[1]

        query = query.view(batch_size, -1, attn.heads, head_dim).transpose(1, 2)

        key = key.view(batch_size, -1, attn.heads, head_dim).transpose(1, 2)
        value = value.view(batch_size, -1, attn.heads, head_dim).transpose(1, 2)

        if attn.norm_q is not None:
            query = attn.norm_q(query)
        if attn.norm_k is not None:
            key = attn.norm_k(key)

        # the output of sdp = (batch, num_heads, seq_len, head_dim)
        # TODO: add support for attn.scale when we move to Torch 2.1
        hidden_states = torch.nn.functional.scaled_dot_product_attention(
            query, key, value, attn_mask=attention_mask, dropout_p=0.0, is_causal=False
        )

        hidden_states = hidden_states.transpose(1, 2).reshape(batch_size, -1, attn.heads * head_dim)
        hidden_states = hidden_states.to(query.dtype)

        # linear proj
        hidden_states = attn.to_out[0](hidden_states)
        # dropout
        hidden_states = attn.to_out[1](hidden_states)

        if input_ndim == 4:
            hidden_states = hidden_states.transpose(-1, -2).reshape(batch_size, channel, height, width)

        if attn.residual_connection:
            hidden_states = hidden_states + residual

        hidden_states = hidden_states / attn.rescale_output_factor

        return hidden_states

In [ ]:
class PeftModel:
    def __init__(self, unet:UNet2DConditionModel, lora_config:dict=None):
        self.unet = unet
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.dtype = torch.float16
        # lora configuration
        if lora_config is not None:
            self.lora_rank = lora_config['lora_rank']
            self.lora_alpha = lora_config['lora_alpha']
            self.lora_dropout = lora_config['lora_dropout']
        else:
            self.lora_rank = 4
            self.lora_alpha = 32
            self.lora_dropout = 0.1

        self.lora_hook()  # register hooks and mount lora matrix

    def lora_hook(self):
        # Downblocks
        for i, downblock in enumerate(self.unet.down_blocks):
            if hasattr(downblock, 'attentions'):
                for attn_module in downblock.attentions:
                    for trfms_module in attn_module.transformer_blocks:
                        # list of BasicTransformerBlock
                        trfms_module.attn2.to_q = LoRALinear(trfms_module.attn2.to_q, self.lora_rank, self.lora_alpha, self.lora_dropout).to(self.device, self.dtype)
                        trfms_module.attn2.to_k = LoRALinear(trfms_module.attn2.to_k, self.lora_rank, self.lora_alpha, self.lora_dropout).to(self.device, self.dtype)
                        trfms_module.attn2.to_v = LoRALinear(trfms_module.attn2.to_v, self.lora_rank, self.lora_alpha, self.lora_dropout).to(self.device, self.dtype)
        # Midblocks
        if hasattr(self.unet.mid_block, 'attentions'):
            for attn_module in self.unet.mid_block.attentions:
                for trfms_module in attn_module.transformer_blocks:
                    # list of BasicTransformerBlock
                    trfms_module.attn2.to_q = LoRALinear(trfms_module.attn2.to_q, self.lora_rank, self.lora_alpha, self.lora_dropout).to(self.device, self.dtype)
                    trfms_module.attn2.to_k = LoRALinear(trfms_module.attn2.to_k, self.lora_rank, self.lora_alpha, self.lora_dropout).to(self.device, self.dtype)
                    trfms_module.attn2.to_v = LoRALinear(trfms_module.attn2.to_v, self.lora_rank, self.lora_alpha, self.lora_dropout).to(self.device, self.dtype)
        # Upblocks
        for i, upblock in enumerate(self.unet.up_blocks):
            if hasattr(upblock, 'attentions'):
                for attn_module in upblock.attentions:
                    for trfms_module in attn_module.transformer_blocks:
                        # list of BasicTransformerBlock
                        trfms_module.attn2.to_q = LoRALinear(trfms_module.attn2.to_q, self.lora_rank, self.lora_alpha, self.lora_dropout).to(self.device, self.dtype)
                        trfms_module.attn2.to_k = LoRALinear(trfms_module.attn2.to_k, self.lora_rank, self.lora_alpha, self.lora_dropout).to(self.device, self.dtype)
                        trfms_module.attn2.to_v = LoRALinear(trfms_module.attn2.to_v, self.lora_rank, self.lora_alpha, self.lora_dropout).to(self.device, self.dtype)

    
    def __getattr__(self, name):
        return getattr(self.unet, name)
    

    def train(self, t_ddim:int):
        optimizer = torch.optim.Adam(self.unet.parameters(), lr=self.lr)
        datas = datasetlib.select()  # load datasets
        for epoch in range(self.epoches):
            t = difflib.calc_t(t_ddim)
            print(f'training epoch {epoch}')
            for data in datas:
                print(data)
                # preprocess
                prompt = data['prompt']
                img_path = data['image_path']
                ann_path = data['annotation_path']
                with torch.no_grad():
                    image = difflib.read_image(img_path, 'Tensor').to(self.device)
                    latent = difflib.encode_image(image)
                    noise, noise_latent = difflib.add_noise(latent, t)
                    embedding = difflib.encode_prompt(prompt)
                    with open(ann_path, 'rb') as f:
                        labels = pickle.load(f)  # labels is a list of tuple like (token_index, seg_image)
                    labels = [(index, utils.resample_image_tensor(torch.where(mask == 1, 0.9, 0.1))) for index, mask in labels]  # use softmask&resize to (64, 64)
                # finetune
                noise_pred, latent_pred = self.predict(noise_latent, t, embedding)
                attn_maps = self.stable_diffusion.attn_scores
                print(len(attn_maps))
                attn_losses = []
                for attn_map in attn_maps:  # choose different attention layers, from 64*64*77 to 8*8*77
                    # attention map shaped like (77, n, n), resize it to (77, 64, 64) and use the token_index ones for BCE loss
                    print(attn_map.shape)
                    for index, mask in labels:
                        mask = mask.to(self.device)
                        resized_attn_map = utils.resample_image_tensor(attn_map[index])  # resize attention map to (64, 64)
                        loss = torch.nn.functional.binary_cross_entropy_with_logits(resized_attn_map, mask)
                        attn_losses.append(loss)
                diff_loss = torch.nn.functional.mse_loss(noise_pred, noise)
                loss = torch.stack(attn_losses).sum() + diff_loss
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            print(f'finish training epoch {epoch}')
        print(f'finish tuning')
        lora_state_dict = {k:v for k, v in self.unet.state_dict().items() if 'lora_' in k.lower()}
        torch.save(lora_state_dict, lora_path)
    
    def load(self):
        lora_state_dict = torch.load(lora_path)
        self.unet.load_state_dict(lora_state_dict, strict=False)

In [ ]:
def train(self):
        # preprocess
        print('1/6:     preprocessing data...')
        self.frozen()
        optimizer = torch.optim.Adam(self.peft_model.unet.parameters(), lr=self.lr)
        datas = dataset_utils.select()

        # train
        print('2/6:     start training...')
        train_loss_attnmap, train_loss_diff = [], []
        for epoch in range(self.epoches):
            # randomly choose a timestep for training
            t_ddim = random.choice(self.sample_timesteps)
            t = diffusion_utils.calc_t(self.peft_model.scheduler, t_ddim)
            print(f'2/6-a:    training epoch {epoch}, using ddim timestep {t_ddim}')
            epoch_loss_attnmap, epoch_loss_diff = [], []

            for data in datas:
                # pre-process
                with torch.no_grad():
                    prompt = data['prompt']
                    img_path = data['image_path']
                    ann_path = data['annotation_path']
                    image = diffusion_utils.read_image(img_path, 'Tensor').to(self.device)
                    latent = diffusion_utils.encode_image(self.peft_model.vae, image)
                    noise, noise_latent = diffusion_utils.add_noise(self.peft_model.scheduler, latent, t)
                    embedding = diffusion_utils.encode_prompt(self.peft_model.pipe, prompt)
                    with open(ann_path, 'rb') as f:
                        labels = pickle.load(f)  # labels is a list of tuple like (token_index, seg_image)
                    labels = [(index, diffusion_utils.resample(torch.where(mask == 1, self.soft_mask[0], self.soft_mask[1]).to(self.dtype))) for index, mask in labels]  # resize to (64, 64)
                # finetune
                with torch.autocast(device_type=self.device, dtype=self.dtype):
                    noise_pred = diffusion_utils.predict(self.peft_model.unet, noise_latent, t, embedding)
                    attn_maps = self.peft_model.attn_scores
                    attn_losses = []
                    for i, attn_map in enumerate(attn_maps):  # choose different attention layers, from 64*64*77 to 8*8*77
                        # attention map shaped like (77, n, n), resize it to (77, 64, 64) and use the token_index ones for BCE loss
                        if self.layer_weight[i] == 0:   continue
                        for index, mask in labels:
                            mask = mask.to(self.device)
                            resized_attn_map = diffusion_utils.resample(attn_map[index])    # resize attention map to (64, 64)
                            # calculate T for attention map down scaling, in case sigmoid output reach 1or0 causing gradient disappearence or explosion
                            # calculate pos_weight for mask positive class loss up scaling, because positive class usually only takes up a small proportion
                            # with torch.no_grad():
                            #     maxval = resized_attn_map.abs().max()
                            #     T = max(maxval / 6.0, 1.0)
                            #     num_pos = (mask == self.soft_mask[0]).sum()
                            #     num_neg = (mask == self.soft_mask[1]).sum()
                            #     pos_weight = (num_neg / (num_pos + 1e-6)).to(dtype=self.dtype, device=self.device)
                            # resized_attn_map = resized_attn_map / T
                            loss = torch.nn.functional.binary_cross_entropy_with_logits(resized_attn_map, mask)
                            attn_losses.append(self.layer_weight[i]*loss)
                    diff_loss = torch.nn.functional.mse_loss(noise_pred, noise)
                    attn_loss = torch.stack(attn_losses).sum()
                    loss = self.lambda_attn * attn_loss + diff_loss
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()
                epoch_loss_attnmap.append(attn_loss.item())
                epoch_loss_diff.append(diff_loss.item())
            train_loss_attnmap.append(epoch_loss_attnmap)
            train_loss_diff.append(epoch_loss_diff)
            print(f'2/6-b:    finish training epoch {epoch}')
        print(f'3/6:    finish training')
        
        # save
        print('4/6:     saving lora_weights...')
        lora_state_dict = {k:v for k, v in self.peft_model.unet.state_dict().items() if 'lora_' in k.lower()}
        torch.save(lora_state_dict, self.lora_path)
        self.peft_model.unet.eval()
        with open(os.path.join(self.folder_path, 'diff_loss.pkl'), 'wb') as f:
            pickle.dump(train_loss_diff, f)  # list of list of every data loss in an epoch, like [[0.8, 0.7, ...], [...], ...]
        with open(os.path.join(self.folder_path, 'attnmap_loss.pkl'), 'wb') as f:
            pickle.dump(train_loss_attnmap, f)
        visualize_utils.visualize_epoch_loss(train_loss_diff, os.path.join(self.folder_path, 'diff_loss.png'))
        visualize_utils.visualize_epoch_loss(train_loss_attnmap, os.path.join(self.folder_path, 'attnmap_loss.png'))

        print('6/6:     successfully finish fine-tuning!')

In [ ]:
class DiffusionLoraRLTrainer:
    def __init__(
        self,
        peft_model:StableDiffusion,
        mask2former:Mask2Former,
        lora_path:str,
        folder_path:str,

        epoches:int=1,
        batch_size:int=4,
        lr:float=1e-6,
        gamma:float=0.99,
        
        layer_weight:list[int]=[0.05, 0,   0.1, 0.05,   0.2, 0.15,   0.2, 0.15, 0.1,   0.1, 0.05, 0,  0.05, 0, 0,   0.3],
        sample_timesteps:list[int]=[2, 3, 4, 5, 6, 7, 8, 9],
        value_func:Literal['dice', 'jaccard']='dice',
    ):
        # models
        self.peft_model = peft_model
        self.mask2former = mask2former
        self.recorder = DiffusionLoraRLLossRecorder(os.path.join(folder_path, 'RL_loss.png'))
        # hyperparameters
        self.epoches = epoches
        self.batch_size = batch_size
        self.lr = lr
        self.gamma = gamma
        self.dtype = torch.float32
        self.device = 'cuda'
        # specific arguments
        self.lora_path = lora_path
        self.value_func = value_func
        self.layer_weight = layer_weight
        self.sample_timesteps = sample_timesteps

    def frozen(self):
        for name, param in self.peft_model.unet.named_parameters():
            if 'lora_' not in name:
                param.requires_grad = False
    
    @torch.no_grad()
    def merge_attnmap(self, token_index:int, attn_scores:list[torch.Tensor]):
        # attn_scores shaped as [(n, v, v), ...], which has a length of num_layers 16
        merge_score = torch.zeros((64, 64), dtype=self.dtype, device=self.device)
        for i, attn_score in enumerate(attn_scores):
            token_attn_score = attn_score[token_index]
            if token_attn_score.shape != (64, 64):
                token_attn_score = diffusion_utils.resample(token_attn_score)
            merge_score += self.layer_weight[i]*token_attn_score
        return merge_score
    
    @torch.no_grad()
    def train(self):
        self.frozen()
        optimizer = torch.optim.Adam(self.peft_model.unet.parameters(), lr=self.lr)
        datas = dataset_utils.select()
        dataset = DiffusionLoraRLDataset(datas, self.dtype, self.device)
        loader = DataLoader(dataset, self.batch_size, shuffle=True, collate_fn=collate_fn)
        # acquire s, V(s), a, s', V(s'), R then acquire (A, a)
        cur_batch = 0
        for i in range(self.epoches):
            for batch in tqdm(loader, desc='training batches'):
                cur_batch += 1
                images = batch['images']    
                prompts = batch['prompts']
                labels = batch['labels']
                gts = batch['gts']
                batch_size = len(prompts)
                t_ddim = random.choice(self.sample_timesteps)
                t = diffusion_utils.calc_t(self.peft_model.scheduler, t_ddim)
                t_prev = diffusion_utils.calc_t(self.peft_model.scheduler, t_ddim-1)
                latents = diffusion_utils.encode_image(self.peft_model.vae, images)

                # s:noise_latents
                noise, noise_latents = diffusion_utils.add_noise(self.peft_model.scheduler, latents, t)
                embeddings = diffusion_utils.encode_prompt(self.peft_model.pipe, prompts)
                del latents

                # a:noise_preds
                with torch.enable_grad():
                    # this would set noise_preds, log_prob, attn_scores as `requires_grad` variable, with addtional process to noise_preds and attn_scores, only log_prob would be saved with grad_on
                    noise_preds = diffusion_utils.predict(self.peft_model.unet, noise_latents, t, embeddings)
                    dist = torch.distributions.Normal(loc=noise_preds, scale=torch.ones_like(noise_preds))
                    sample_actions = dist.rsample()
                    log_probs = dist.log_prob(sample_actions).sum(dim=(1, 2, 3))
                noise_preds = sample_actions

                # V(s):attention map loss
                attn_scores = [attn_score.detach() for attn_score in self.peft_model.attn_scores]
                v_status = []
                for b in range(batch_size):
                    label_b = labels[b]  # list of tuple(index, mask)
                    attn_scores_b = [attn_score[b] for attn_score in attn_scores]
                    v_s = 0
                    for index, mask in label_b: 
                        merge_score = self.merge_attnmap(index, attn_scores_b)
                        if self.value_func == 'dice':
                            v_s += dice_value(merge_score, mask).item()
                        elif self.value_func == 'jaccard':
                            v_s += jaccard_value(merge_score, mask).item()
                    v_status.append(v_s)  # list of float
                del attn_scores

                # s':next_latents
                next_latents = diffusion_utils.step(self.peft_model.scheduler, noise_latents, t, noise_preds)
                
                # V(s'):next_latents attention map loss
                next_noise_preds = diffusion_utils.predict(self.peft_model.unet, next_latents, t_prev, embeddings)
                attn_scores = self.peft_model.attn_scores
                v_next_status = []
                for b in range(batch_size):
                    label_b = labels[b]  # list of tuple(index, mask)
                    attn_scores_b = [attn_score[b] for attn_score in attn_scores]
                    v_next_s = 0
                    for index, mask in label_b: 
                        merge_score = self.merge_attnmap(index, attn_scores_b)
                        if self.value_func == 'dice':
                            v_next_s += dice_value(merge_score, mask).item()
                        elif self.value_func == 'jaccard':
                            v_next_s += jaccard_value(merge_score, mask).item()
                    v_next_status.append(v_next_s)
                del attn_scores

                # R:mIou(s') - mIou(s)
                rewards = []
                img_s = diffusion_utils.decode_latent(self.peft_model.vae, diffusion_utils.origin(self.peft_model.scheduler, noise_latents, t, noise_preds))  # (batch_size, 512, 512, 3)
                img_next_s = diffusion_utils.decode_latent(self.peft_model.vae, diffusion_utils.origin(self.peft_model.scheduler, next_latents, t_prev, next_noise_preds))
                seg_s = self.mask2former(img_s)
                seg_next_s = self.mask2former(img_next_s)
                del img_s, img_next_s
                for i in range(batch_size):   # both gts and seg_s is list of torch.Tensor shaped as (512, 512)
                    miou_s = mIou(seg_s[i], gts[i].to(self.device))
                    miou_next_s = mIou(seg_next_s[i], gts[i].to(self.device))
                    rewards.append(miou_next_s - miou_s)
                del noise_preds, next_noise_preds, seg_s, seg_next_s

                # loss: log_prob * [ γV(s') + R - V(s) ]
                batch_loss = 0
                with torch.enable_grad():
                    rl_loss = []
                    for i in range(batch_size):
                        adv = v_next_status[i] * self.gamma + rewards[i] - v_status[i]
                        loss = - adv * log_probs[i]
                        rl_loss.append(loss)
                    loss = torch.stack(rl_loss).sum()
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()
                    batch_loss += loss.item()
                self.recorder.update(batch_loss, cur_batch)
            # finish a epoch
        # finish training
        lora_state_dict = {k:v for k, v in self.peft_model.unet.state_dict().items() if 'lora_' in k.lower()}
        torch.save(lora_state_dict, self.lora_path)
        self.peft_model.unet.eval()

In [ ]:
def critic(self, merged_attn_scores:torch.Tensor, target_indexs:list[list[int]])->torch.Tensor:
    assert merged_attn_scores.ndim == 4  #(batch_size, num_tokens, h, w)
    value_scores = []
    for i in range(merged_attn_scores.shape[0]):
        token_attn_scores = merged_attn_scores[i, target_indexs[i]]  # (target_tokens, h, w)
        token_value_scores = self.critic_net(token_attn_scores)  # input target_tokens as batch
        value_scores.append(token_value_scores)
    value_scores = torch.stack(value_scores).to(dtype=self.dtype, device=self.device)
    return value_scores  # (batch_size, target_tokens, )

def frozen_all(self):
    for name, param in self.peft_model.unet.named_parameters():
        param.requires_grad = False

def train_critic(self):
    self.frozen_all()
    optimizer = torch.optim.Adam(self.critic_net.parameters(), lr=self.lr)
    lr_scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=self.lr_decay)
    datas = dataset_utils.select()
    dataset = DiffusionLoraRLDataset(datas, self.dtype, self.device)
    loader = DataLoader(dataset, self.batch_size, shuffle=True, collate_fn=collate_fn)
    # loss = R + gamma * V(s') - V(s)



def train(self):
    self.frozen()
    optimizer = torch.optim.Adam([p for p in self.peft_model.unet.parameters() if p.requires_grad], lr=self.lr)
    lr_scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=self.lr_decay)
    datas = dataset_utils.select()
    dataset = DiffusionLoraRLDataset(datas, self.dtype, self.device)
    loader = DataLoader(dataset, self.batch_size, shuffle=True, collate_fn=collate_fn)
    # acquire s, V(s), a, s', V(s'), R then acquire (A, a)
    cur_batch = 0
    for i in range(self.epoches):
        for batch in tqdm(loader, desc='training batches'):
            cur_batch += 1
            images = batch['images']
            prompts = batch['prompts']
            indexs = batch['indexs']
            with torch.no_grad():
                t_ddim = random.choice(self.sample_timesteps)
                t = diffusion_utils.calc_t(self.peft_model.scheduler, t_ddim)
                t_prev = diffusion_utils.calc_t(self.peft_model.scheduler, t_ddim - 1)
                embeddings = diffusion_utils.encode_prompt(self.peft_model.pipe, prompts)
                latents = diffusion_utils.encode_image(self.peft_model.vae, images)
                noise, noise_latents = diffusion_utils.add_noise(self.peft_model.scheduler, latents, t)
                del latents, noise
                torch.cuda.empty_cache()
            # KL: kl-regularzation
                disable_lora(self.peft_model)
                noise_preds_kl = diffusion_utils.predict(self.peft_model.unet, noise_latents, t, embeddings)
                enable_lora(self.peft_model)
            # a:logprobs
            noise_preds = diffusion_utils.predict(self.peft_model.unet, noise_latents, t, embeddings)
            next_latents, logprobs = diffusion_utils.step_logprob(self.peft_model.scheduler, noise_latents, t, noise_preds, t_prev, self.eta)
            kl_loss = torch.nn.functional.mse_loss(noise_preds, noise_preds_kl)
            with torch.no_grad():
            # R_t:clip cosine similarity
                origin_images = diffusion_utils.reformat_image(diffusion_utils.decode_latent(self.peft_model.vae, diffusion_utils.origin(self.peft_model.scheduler, noise_latents, t, noise_preds)))  # batch_size length of PIL.Image
                reward_scores = self.clip(prompts, origin_images)
                del origin_images
                torch.cuda.empty_cache()
            # V(s):attention map score
                merged_attn_scores = self.peft_model.merged_attn_scores
                v_s = self.critic(merged_attn_scores, indexs)
                del merged_attn_scores
                torch.cuda.empty_cache()
            # V(s'):attention map scores
                noise_preds = diffusion_utils.predict(self.peft_model.unet, next_latents, t_prev, embeddings)
                merged_attn_scores = self.peft_model.merged_attn_scores
                v_s_prev = self.critic(merged_attn_scores, indexs)
                del merged_attn_scores
                torch.cuda.empty_cache()
            # R_t_prev:clip cosine similarity
                origin_images_prev = diffusion_utils.reformat_image(diffusion_utils.decode_latent(self.peft_model.vae, diffusion_utils.origin(self.peft_model.scheduler, next_latents, t_prev, noise_preds)))
                reward_scores_prev = self.clip(prompts, origin_images_prev)
                del origin_images_prev, noise_preds
                torch.cuda.empty_cache()
            # R:reward improvement
                rewards = reward_scores_prev - reward_scores
            # A: advantage score
                adv = self.lambda_reward * rewards + self.gamma * v_s_prev - v_s
                del v_s, v_s_prev, reward_scores, reward_scores_prev
                torch.cuda.empty_cache()
            loss = (- adv * logprobs).sum() + self.lambda_kl * kl_loss
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            # finish a batch
            self.recorder_loss.update(loss.item(), cur_batch)
            self.recorder_adv.update(adv.sum().item(), cur_batch)
            self.recorder_kl.update(kl_loss.item(), cur_batch)
        # finish an epoch
        lr_scheduler.step()
    # finish training
    lora_state_dict = {k:v for k, v in self.peft_model.unet.state_dict().items() if 'lora_' in k.lower()}
    torch.save(lora_state_dict, self.lora_path)
    self.peft_model.unet.eval()

In [ ]:
def train(self):
        # preprocess
        self.frozen()
        optimizer = torch.optim.Adam(self.peft_model.unet.parameters(), lr=self.lr)
        lr_scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=self.lr_decay)
        datas = dataset_utils.select()
        dataset = DiffusionLoraSFTDataset(datas, self.soft_mask, self.dtype, self.device)
        loader = DataLoader(dataset, self.batch_size, shuffle=True, collate_fn=collate_fn)
        # train
        cur_batch = 0
        for epoch in range(self.epoches):
            for batch in tqdm(loader, desc='training batches'):
                cur_batch += 1
                prompts = batch['prompts']
                images = batch['images']
                labels = batch['labels']
                t_ddim = random.choice(self.sample_timesteps)   # randomly choose a timestep for training
                t = diffusion_utils.calc_t(self.peft_model.scheduler, t_ddim)
                with torch.no_grad():
                    latents = diffusion_utils.encode_image(self.peft_model.vae, images)
                    _, noise_latents = diffusion_utils.add_noise(self.peft_model.scheduler, latents, t)
                    embeddings = diffusion_utils.encode_prompt(self.peft_model.pipe, prompts)
                    disable_lora(self.peft_model)
                    noise_preds_kl = diffusion_utils.predict(self.peft_model.unet, noise_latents, t, embeddings)
                    enable_lora(self.peft_model)
                    del latents
                # forward
                noise_preds = diffusion_utils.predict(self.peft_model.unet, noise_latents, t, embeddings)
                attn_scores = self.peft_model.attn_scores  # list of shaped like (batch_size, n, v, v)
                attn_losses, attn_weights = [], []
                # calc_loss
                for i, attn_score in enumerate(attn_scores):
                    if self.layer_weight[i] == 0: continue
                    for b in range(len(prompts)):
                        label_b = labels[b]
                        for index, mask in label_b:
                            mask = mask.to(self.device)
                            resized_attn_score = diffusion_utils.resample(attn_score[b][index])
                            if self.loss_func == 'dice':
                                loss = 0.5*dice_loss(resized_attn_score, mask) + 0.5*bce_loss(resized_attn_score, mask)
                            elif self.loss_func == 'jaccard':
                                loss = 0.5*jaccard_loss(resized_attn_score, mask) + 0.5*bce_loss(resized_attn_score, mask)
                            attn_losses.append(loss)
                            # calculate extra weight to prevent large mask cause lora overfit
                            weight = (1 / ((mask == self.soft_mask[0]).sum() + 1e-8)).detach() * self.layer_weight[i]
                            attn_weights.append(weight)  # list torch.Tensor(x), tensor dim=0
                # norm_weights
                attn_weights = torch.stack(attn_weights)
                attn_weights = attn_weights / attn_weights.sum()
                # sum_loss
                diff_loss = torch.nn.functional.mse_loss(noise_preds, noise_preds_kl)
                attn_loss = (torch.stack(attn_losses)*attn_weights).sum()
                # loss = self.lambda_attn * attn_loss + diff_loss
                loss = attn_loss
                # step
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                # finish a batch
                self.diff_recorder.update(diff_loss.item(), cur_batch)
                self.attn_recorder.update(attn_loss.item(), cur_batch)
            # finish all batch/finish a epoch
            lr_scheduler.step()
        # finish training
        lora_state_dict = {k:v for k, v in self.peft_model.unet.state_dict().items() if 'lora_' in k.lower()}
        torch.save(lora_state_dict, self.lora_path)
        self.peft_model.unet.eval()

In [ ]:
class CriticNet(torch.nn.Module):
    def __init__(self,
        base_channels:int=32,
        up_channels:int=64,
        pooling_dims:int=8,
        hidden_states:int=256,
    ):
        super().__init__()
        self.conv_layers = torch.nn.Sequential(
            torch.nn.Conv2d(1, base_channels, kernel_size=3, padding=1),
            torch.nn.SiLU(),
            torch.nn.Conv2d(base_channels, up_channels, kernel_size=3, padding=1),
            torch.nn.SiLU(),
            torch.nn.AdaptiveAvgPool2d(pooling_dims),
            torch.nn.Flatten()
        )
        self.fc_layers = torch.nn.Sequential(
            torch.nn.Linear(up_channels * pooling_dims * pooling_dims, hidden_states),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden_states, 1)
        )

    def forward(self, x:torch.Tensor):
        # x shaped as (batch_size, h, w)
        x = x.unsqueeze(1)
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        x = x.squeeze(-1)
        return x

In [ ]:
from sdlib import diffusion_utils, dataset_utils
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from model.stable_diffusion import StableDiffusion
from model.clip import CLIP
from config.default_config import LAYER_WEIGHT, SAMPLE_TIMESTEPS_RL_REGULAR
from lora.peft_model import disable_lora, enable_lora
import random
import torch
from torch.utils.data import DataLoader
import os
from typing import Any
from tqdm import tqdm


def collate_fn(batch):
    prompts = [item['prompt'] for item in batch]  # list[str]
    return {
        'prompts':prompts,
    }

class DiffusionLoraRLLossRecorder:
    def __init__(self, save_path:str, save_gap:int=50):
        self.losses = []
        self.steps = []
        self.save_path = save_path
        self.save_gap = save_gap  # every `save_gap` batch refresh image

    def update(self, loss:float, step:int):
        if step % self.save_gap == 0:
            self.losses.append(loss)
            self.steps.append(step)
            self.save_plot()

    def save_plot(self):
        plt.figure()
        plt.plot(self.steps, self.losses, marker='o', linestyle='-', color='b', label='loss')
        plt.xlabel("batch")
        plt.ylabel("loss")
        plt.title("Reinforcement Learning Batch Training Loss")
        plt.legend()
        plt.grid(True)
        plt.savefig(self.save_path, dpi=300)
        plt.close()



class DiffusionLoraRLDataset(torch.utils.data.Dataset):
    def __init__(self, datas:list[dict[str, Any]], dtype, device):
        super().__init__()
        self.dtype = dtype
        self.device = device
        
        self.datas = datas
    
    def __getitem__(self, index):
        data = self.datas[index]
        prompt = data['prompt']
        return {
            'prompt':prompt,
        }

    def __len__(self):
        return len(self.datas)
    





class DiffusionLoraRLCLIPTrainer:
    def __init__(
        self,
        peft_model:StableDiffusion,
        clip:CLIP,
        lora_path:str,
        folder_path:str,

        epoches:int=3,
        batch_size:int=4,
        lr:float=1e-4,
        lr_decay:float=0.95,
        gamma:float=0.95,
        eta:float=1.0,

        lambda_reward:float=5.0,
        lambda_kl:float=0.2,
        sample_timesteps:list[int]=SAMPLE_TIMESTEPS_RL_REGULAR,
        layer_weight:list[float]=LAYER_WEIGHT,
    ):
        self.peft_model = peft_model
        self.clip = clip
        self.recorder = DiffusionLoraRLLossRecorder(os.path.join(folder_path, 'RL_clip_loss.png'))
        # hyperparameters
        self.epoches = epoches
        self.batch_size = batch_size
        self.lr = lr
        self.lr_decay = lr_decay
        self.gamma = gamma
        self.eta = eta
        self.dtype = torch.float32
        self.device = 'cuda'
        # specific arguments
        self.lora_path = lora_path
        self.lambda_kl = lambda_kl
        self.lambda_reward = lambda_reward
        self.layer_weight = layer_weight
        self.sample_timesteps = sample_timesteps

    
    def frozen(self):
        for name, param in self.peft_model.unet.named_parameters():
            if 'lora_' not in name:
                param.requires_grad = False
    


    
    def reward(self, prompts:list[str], latents:torch.Tensor, t:int, noise_preds:torch.Tensor)->torch.Tensor:
        origin_latents = diffusion_utils.origin(self.peft_model.scheduler, latents, t, noise_preds)
        origin_images = diffusion_utils.decode_latent(self.peft_model.vae, origin_latents)
        origin_images = diffusion_utils.reformat_image(origin_images)
        rewards = self.clip(prompts, origin_images)
        return rewards  # (batch_size, )
    
 

    def train(self):
        self.frozen()
        optimizer = torch.optim.Adam([p for p in self.peft_model.unet.parameters() if p.requires_grad], lr=self.lr)
        lr_scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=self.lr_decay)
        datas = dataset_utils.select()
        dataset = DiffusionLoraRLDataset(datas, self.dtype, self.device)
        loader = DataLoader(dataset, self.batch_size, shuffle=True, collate_fn=collate_fn)
        # acquire R for each s
        cur_batch = 0
        for i in range(self.epoches):
            for batch in tqdm(loader, desc=f'training epoch{i+1}'):
                cur_batch += 1
                prompts = batch['prompts']
                with torch.no_grad():
                    t_ddim = random.choice(self.sample_timesteps)
                    t = diffusion_utils.calc_t(self.peft_model.scheduler, t_ddim)
                    t_prev = diffusion_utils.calc_t(self.peft_model.scheduler, t_ddim-1)
                    embeddings = diffusion_utils.encode_prompt(self.peft_model.pipe, prompts)
                    latents = diffusion_utils.latent_t_comb(self.peft_model, embeddings, t_ddim)
                # KL: kl-regularzation
                    disable_lora(self.peft_model)
                    noise_preds_kl = diffusion_utils.predict(self.peft_model.unet, latents, t, embeddings)
                    enable_lora(self.peft_model)
                # a:logprobs
                noise_preds = diffusion_utils.predict(self.peft_model.unet, latents, t, embeddings)
                _, logprobs = diffusion_utils.step_logprob(self.peft_model.scheduler, latents, t, noise_preds, t_prev, self.eta)
                kl_loss = torch.nn.functional.mse_loss(noise_preds, noise_preds_kl)
                with torch.no_grad():
                # R:rewards
                    rewards = self.reward(prompts, latents, t, noise_preds)
                torch.cuda.empty_cache()
                # step
                loss = (- rewards * logprobs).sum() + self.lambda_kl * kl_loss
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                # finish a batch
                self.recorder.update(loss.item(), cur_batch)
            # finish an epoch
            lr_scheduler.step()
        # finish training
        lora_state_dict = {k:v for k, v in self.peft_model.unet.state_dict().items() if 'lora_' in k.lower()}
        torch.save(lora_state_dict, self.lora_path)
        self.peft_model.unet.eval()

In [ ]:
from model.stable_diffusion import StableDiffusion
from model.mask2former import Mask2Former
from model.clip import CLIP
from sdlib import diffusion_utils, visualize_utils, dataset_utils
from lora.peft_model import get_peft_model, load_lora_weight, disable_lora, enable_lora
from lora.lora_sft import DiffusionLoraSFTTrainer
from lora.lora_rl_regular_reward import DiffusionLoraRLRegularTrainer
from lora.lora_rl_clip_reward import DiffusionLoraRLCLIPTrainer
from lora.lora_rl_m2f_reward import DiffusionLoraRLM2FTrainer
from config.default_config import *
from config.training_config import *
import os
import torch
import pickle
import math


supervised_infer_test = 0
nonsupervised_infer_test = 0
output_test = 1
reverse_test = 0
test = 0

download_dataset = 0

score_mask2former_test = 0
score_clip_test = 0
score_regular_test = 0

sft_train = 0
rl_clip_train = 0
rl_regular_train = 0
rl_mask2former_train = 0


def lora_gen(random_noise, prompt, lora_path, save_folder:str, name:str):
    print(f'generating {name}...')
    stable_diffusion = StableDiffusion()
    get_peft_model(stable_diffusion)
    t_ddim = 50
    hook_lora_step = t_ddim - max(SAMPLE_TIMESTEPS)
    remove_lora_step = t_ddim - min(SAMPLE_TIMESTEPS)
    tokens = diffusion_utils.get_tokens(stable_diffusion.tokenizer, prompt)
    embeddings = diffusion_utils.encode_prompt(stable_diffusion.pipe, prompt)
    noise_latent = random_noise
    stable_diffusion.scheduler.ets = []
    stable_diffusion.scheduler.counter = 0
    for i, t in enumerate(stable_diffusion.scheduler.timesteps[-t_ddim:]):
        if i == hook_lora_step-1:
            load_lora_weight(stable_diffusion, lora_path)
        if i == remove_lora_step-1:
            disable_lora(stable_diffusion)
        noise_pred = diffusion_utils.predict(stable_diffusion.unet, noise_latent, t, embeddings)
        latent_pred = diffusion_utils.step(stable_diffusion.scheduler, noise_latent, t, noise_pred)
        noise_latent = latent_pred

    img = diffusion_utils.decode_latent(stable_diffusion.vae, noise_latent)[0]
    attn_scores = stable_diffusion.attn_scores
    path_lora_img = os.path.join(save_folder, name+'.png')
    path_lora_attnmap = os.path.join(save_folder, name + '_attnmap.png')

    visualize_utils.visualize_image(img, path_lora_img)
    visualize_utils.visualize_merged_attn_score(attn_scores, tokens, save_path=path_lora_attnmap)


def gen(random_noise, prompt, save_folder:str, name:str='original'):
    print(f'generating {name}...')
    stable_diffusion = StableDiffusion()
    t_ddim = 50
    tokens = diffusion_utils.get_tokens(stable_diffusion.tokenizer, prompt)
    embeddings = diffusion_utils.encode_prompt(stable_diffusion.pipe, prompt)
    noise_latent = random_noise
    stable_diffusion.scheduler.ets = []
    stable_diffusion.scheduler.counter = 0
    for i, t in enumerate(stable_diffusion.scheduler.timesteps[-t_ddim:]):
        noise_pred = diffusion_utils.predict(stable_diffusion.unet, noise_latent, t, embeddings)
        latent_pred = diffusion_utils.step(stable_diffusion.scheduler, noise_latent, t, noise_pred)
        noise_latent = latent_pred

    img = diffusion_utils.decode_latent(stable_diffusion.vae, noise_latent)[0]
    attn_scores = stable_diffusion.attn_scores
    path_lora_img = os.path.join(save_folder, name+'.png')
    path_lora_attnmap = os.path.join(save_folder, name + '_attnmap.png')

    visualize_utils.visualize_image(img, path_lora_img)
    visualize_utils.visualize_merged_attn_score(attn_scores, tokens, save_path=path_lora_attnmap)


def lora_rev(img_path, prompt, t_ddim, lora_path, save_folder, name):
    print(f'reversing {name}...')
    img = diffusion_utils.read_image(img_path, 'Tensor', unsqueeze=True) # convert to batch
    stable_diffusion = StableDiffusion()
    get_peft_model(stable_diffusion)
    latent = diffusion_utils.encode_image(stable_diffusion.vae, img)
    _, noise_latent = diffusion_utils.add_noise(stable_diffusion.scheduler, latent, diffusion_utils.calc_t(stable_diffusion.scheduler, t_ddim))
    hook_lora_step = t_ddim - max(SAMPLE_TIMESTEPS)
    remove_lora_step = t_ddim - min(SAMPLE_TIMESTEPS)
    tokens = diffusion_utils.get_tokens(stable_diffusion.tokenizer, prompt)
    embeddings = diffusion_utils.encode_prompt(stable_diffusion.pipe, prompt)
    stable_diffusion.scheduler.ets = []
    stable_diffusion.scheduler.counter = 0
    if hook_lora_step <= 0:
        load_lora_weight(stable_diffusion, lora_path)
    for i, t in enumerate(stable_diffusion.scheduler.timesteps[-t_ddim:]):
        if i == hook_lora_step-1:
            load_lora_weight(stable_diffusion, lora_path)
        if i == remove_lora_step-1:
            disable_lora(stable_diffusion)
        noise_pred = diffusion_utils.predict(stable_diffusion.unet, noise_latent, t, embeddings)
        latent_pred = diffusion_utils.step(stable_diffusion.scheduler, noise_latent, t, noise_pred)
        noise_latent = latent_pred

    img = diffusion_utils.decode_latent(stable_diffusion.vae, noise_latent)[0]
    attn_scores = stable_diffusion.attn_scores
    path_lora_img = os.path.join(save_folder, name+'.png')
    path_lora_attnmap = os.path.join(save_folder, name + '_attnmap.png')

    visualize_utils.visualize_image(img, path_lora_img)
    visualize_utils.visualize_merged_attn_score(attn_scores, tokens, save_path=path_lora_attnmap)

def rev(img_path, prompt, t_ddim, save_folder, name='original'):
    print(f'reversing {name}...')
    img = diffusion_utils.read_image(img_path, 'Tensor', unsqueeze=True)  # convert to batch
    stable_diffusion = StableDiffusion()
    latent = diffusion_utils.encode_image(stable_diffusion.vae, img)
    _, noise_latent = diffusion_utils.add_noise(stable_diffusion.scheduler, latent, diffusion_utils.calc_t(stable_diffusion.scheduler, t_ddim))
    tokens = diffusion_utils.get_tokens(stable_diffusion.tokenizer, prompt)
    embeddings = diffusion_utils.encode_prompt(stable_diffusion.pipe, prompt)
    stable_diffusion.scheduler.ets = []
    stable_diffusion.scheduler.counter = 0
    for i, t in enumerate(stable_diffusion.scheduler.timesteps[-t_ddim:]):
        noise_pred = diffusion_utils.predict(stable_diffusion.unet, noise_latent, t, embeddings)
        latent_pred = diffusion_utils.step(stable_diffusion.scheduler, noise_latent, t, noise_pred)
        noise_latent = latent_pred

    img = diffusion_utils.decode_latent(stable_diffusion.vae, noise_latent)[0]
    attn_scores = stable_diffusion.attn_scores
    path_lora_img = os.path.join(save_folder, name+'.png')
    path_lora_attnmap = os.path.join(save_folder, name + '_attnmap.png')

    visualize_utils.visualize_image(img, path_lora_img)
    visualize_utils.visualize_merged_attn_score(attn_scores, tokens, save_path=path_lora_attnmap)


if reverse_test:
    save_folder = REVERSE_FOLDER_PATH
    prompt = 'A man with red helmet riding motorbike on the dirt road in countryside.'
    t_ddim = 20
    with torch.no_grad():
        img_path = DEMO_PATH
        rev(img_path, prompt, t_ddim, save_folder)
        lora_rev(img_path, prompt, t_ddim, LORA_SFT_PATH, save_folder, 'sft')
        lora_rev(img_path, prompt, t_ddim, LORA_RL_CLIP_PATH, save_folder, 'rl_clip')
        lora_rev(img_path, prompt, t_ddim, LORA_RL_REGULAR_PATH, save_folder, 'rl_regular')
        lora_rev(img_path, prompt, t_ddim, LORA_RL_M2F_PATH, save_folder, 'rl_m2f')

if output_test:
    save_folder = OUTPUT_FOLDER_PATH
    prompt = 'A cat, A dog and a man'
    # prompt = 'A cat and A dog'
    # prompt = 'cat, traffic light, dining table, baseball bat'
    prompt = 'A man eating meals with fork on dining table, a potted plant on the corner'
    # prompt = 'A bookshelf stacked with books, a golden cat laying in sofa, a clock on the wall, a potted plant in the corner, a cosmic landscape photo hanging on the wall'
    with torch.no_grad():
        random_noise = torch.randn((1, 4, 64, 64), dtype=torch.float32, device='cuda')
        random_noise_path = os.path.join(save_folder, 'random_noise.pt')
        torch.save(random_noise, random_noise_path)
        gen(random_noise, prompt, save_folder)
        lora_gen(random_noise, prompt, LORA_SFT_PATH, save_folder, 'sft')
        lora_gen(random_noise, prompt, LORA_RL_CLIP_PATH, save_folder, 'rl_clip')
        lora_gen(random_noise, prompt, LORA_RL_REGULAR_PATH, save_folder, 'rl_regular')
        lora_gen(random_noise, prompt, LORA_RL_M2F_PATH, save_folder, 'rl_m2f')


if test:
    stable_diffusion = StableDiffusion()
    prompt = 'person,traffic light,cat'
    # indexs = [(3, 3), (4, 4), (5, 5)]
    # result = dataset_utils.remap_index(prompt, indexs)
    # print(result)
    tokens = diffusion_utils.get_tokens(stable_diffusion.tokenizer, prompt)
    embeddings = diffusion_utils.encode_prompt(stable_diffusion.pipe, prompt)
    dataset_utils.avg_embeddings(prompt, embeddings)

if rl_mask2former_train:
    stable_diffusion = StableDiffusion()
    mask2former = Mask2Former()
    lora_sft_path = LORA_SFT_PATH
    lora_rl_path = LORA_RL_M2F_PATH
    folder_path = LOSS_FOLDER_PATH

    stable_diffusion = get_peft_model(stable_diffusion)
    load_lora_weight(stable_diffusion, lora_sft_path)
    trainer = DiffusionLoraRLM2FTrainer(stable_diffusion, mask2former, lora_rl_path, folder_path)
    trainer.train()


if rl_clip_train:
    stable_diffusion = StableDiffusion()
    clip = CLIP()
    lora_sft_path = LORA_SFT_PATH
    lora_rl_path = LORA_RL_CLIP_PATH
    folder_path = LOSS_FOLDER_PATH

    stable_diffusion = get_peft_model(stable_diffusion)
    load_lora_weight(stable_diffusion, lora_sft_path)
    trainer = DiffusionLoraRLCLIPTrainer(stable_diffusion, clip, lora_rl_path, folder_path)
    trainer.train()


if download_dataset:
    dataset_utils.download(6001, 10000)

if sft_train:
    stable_diffusion = StableDiffusion()
    lora_path = LORA_SFT_PATH
    folder_path = LOSS_FOLDER_PATH

    stable_diffusion = get_peft_model(stable_diffusion)
    trainer = DiffusionLoraSFTTrainer(stable_diffusion, lora_path, folder_path)
    trainer.train()

if supervised_infer_test:
    stable_diffusion = StableDiffusion()

    path = DEMO_PATH
    lora_path = LORA_SFT_PATH
    img_folder = TEST_FOLDER_PATH
    prompt = 'photo of person motorbike bicycle'
    t_ddim = 8
    tokens = diffusion_utils.get_tokens(stable_diffusion.tokenizer, prompt)

    path_ori_img = os.path.join(img_folder, 'original.png')
    path_ori_attnmap = os.path.join(img_folder, 'original_attnmap.png')
    path_ori_mergemap = os.path.join(img_folder, 'original_merged_attnmap.png')
    img, attn_scores = diffusion_utils.pipeline(stable_diffusion, path, t_ddim, prompt)
    visualize_utils.visualize_merged_attn_score(attn_scores, tokens=tokens, save_path=path_ori_mergemap)
    visualize_utils.visualize_image(img, save_path=path_ori_img)
    visualize_utils.visualize_attn_score(attn_scores[6], tokens=tokens, save_path=path_ori_attnmap)


    stable_diffusion = get_peft_model(stable_diffusion)
    load_lora_weight(stable_diffusion, lora_path)

    path_lora_img = os.path.join(img_folder, 'lora.png')
    path_lora_attnmap = os.path.join(img_folder, 'lora_attnmap.png')
    path_lora_mergemap = os.path.join(img_folder, 'lora_merged_attnmap.png')
    img, attn_scores = diffusion_utils.pipeline(stable_diffusion, path, t_ddim, prompt)
    visualize_utils.visualize_merged_attn_score(attn_scores, tokens=tokens, save_path=path_lora_mergemap)
    visualize_utils.visualize_image(img, save_path=path_lora_img)
    visualize_utils.visualize_attn_score(attn_scores[6], tokens=tokens, save_path=path_lora_attnmap)


if score_mask2former_test:
    mask2former = Mask2Former()
    demo_path = DEMO_PATH
    image = diffusion_utils.read_image(demo_path, 'Image', unsqueeze=True)
    result = mask2former(image)
    for i, r in enumerate(result):
        visualize_utils.visualize_image(r, os.path.join(TEST_FOLDER_PATH, f'trash/{i}.png'))


if rl_regular_train:
    stable_diffusion = StableDiffusion()
    lora_sft_path = LORA_SFT_PATH
    lora_rl_path = LORA_RL_REGULAR_PATH
    folder_path = LOSS_FOLDER_PATH

    stable_diffusion = get_peft_model(stable_diffusion)
    load_lora_weight(stable_diffusion, lora_sft_path)
    trainer = DiffusionLoraRLRegularTrainer(stable_diffusion, lora_rl_path, folder_path)
    trainer.unroll_chain_train()


if nonsupervised_infer_test:
    pure_lora = True
    load_weight = False

    stable_diffusion = StableDiffusion()
    lora_path = LORA_RL_REGULAR_PATH
    lora_path = LORA_SFT_PATH
    # lora_path = LORA_RL_CLIP_PATH
    img_folder = TEST_FOLDER_PATH
    prompt = 'A bookshelf stacked with books, a golden cat laying in sofa, a clock on the wall, a potted plant in the corner, a cosmic landscape photo hanging on the wall'
    # prompt = 'A cat standing next to a dog on the grass, a person stand behind.'
    prompt = 'a cat and a dog and a man'
    # prompt = 'cat, traffic light, motorbike'
    tokens = diffusion_utils.get_tokens(stable_diffusion.tokenizer, prompt)
    random_noise = torch.randn((1, 4, 64, 64), dtype=torch.float32, device='cuda')
    embeddings = diffusion_utils.encode_prompt(stable_diffusion.pipe, prompt)
    t_ddim = 50

    random_noise_path = os.path.join(img_folder, 'random_noise.pt')
    torch.save(random_noise, random_noise_path)

    with torch.no_grad():
        # original output
        noise_latent = random_noise
        for i, t in enumerate(stable_diffusion.scheduler.timesteps[-t_ddim:]):
            print(f'reversing timestep {t} as {i} loop')
            noise_pred = diffusion_utils.predict(stable_diffusion.unet, noise_latent, t, embeddings)
            latent_pred = diffusion_utils.step(stable_diffusion.scheduler, noise_latent, t, noise_pred)
            noise_latent = latent_pred
        
        img = diffusion_utils.decode_latent(stable_diffusion.vae, noise_latent)[0]
        attn_scores = stable_diffusion.attn_scores

        path_ori_img = os.path.join(img_folder, 'test.png')
        path_ori_attnmap = os.path.join(img_folder, 'test_attnmap.png')
        path_ori_mergemap = os.path.join(img_folder, 'test_merged_attnmap.png')
        visualize_utils.visualize_merged_attn_score(attn_scores, tokens=tokens, save_path=path_ori_mergemap)
        visualize_utils.visualize_image(img, save_path=path_ori_img)
        visualize_utils.visualize_attn_score(attn_scores[6], tokens=tokens, save_path=path_ori_attnmap)

        
        
        # output with lora
        if pure_lora:
            t_ddim = 50
            if not load_weight:
                noise_latent = random_noise
            else:
                noise_latent = torch.load(random_noise_path)
            hook_lora_step = t_ddim - max(SAMPLE_TIMESTEPS)
            remove_lora_step = t_ddim - min(SAMPLE_TIMESTEPS)
            stable_diffusion.scheduler.ets = []
            stable_diffusion.scheduler.counter = 0
            for i, t in enumerate(stable_diffusion.scheduler.timesteps[-t_ddim:]):
                print(f'reversing timestep {t} as {i} loop')
                if i == hook_lora_step-1:
                    print('---------------------lora hooked-------------------')
                    stable_diffusion = get_peft_model(stable_diffusion)
                    load_lora_weight(stable_diffusion, lora_path)
                if i == remove_lora_step-1:
                    print('---------------------lora disabled-----------------')
                    disable_lora(stable_diffusion)
                noise_pred = diffusion_utils.predict(stable_diffusion.unet, noise_latent, t, embeddings)
                latent_pred = diffusion_utils.step(stable_diffusion.scheduler, noise_latent, t, noise_pred)
                noise_latent = latent_pred

            img = diffusion_utils.decode_latent(stable_diffusion.vae, noise_latent)[0]
            attn_scores = stable_diffusion.attn_scores

            path_lora_img = os.path.join(img_folder, 'test_lora.png')
            path_lora_attnmap = os.path.join(img_folder, 'test_lora_attnmap.png')
            path_lora_mergemap = os.path.join(img_folder, 'test_lora_merged_attnmap.png')
            visualize_utils.visualize_merged_attn_score(attn_scores, tokens=tokens, save_path=path_lora_mergemap)
            visualize_utils.visualize_image(img, save_path=path_lora_img)
            visualize_utils.visualize_attn_score(attn_scores[6], tokens=tokens, save_path=path_lora_attnmap)

        else:
            latent = noise_latent
            t_ddim = 45
            _, noise_latent = diffusion_utils.add_noise(stable_diffusion.scheduler, noise_latent, diffusion_utils.calc_t(stable_diffusion.scheduler, t_ddim))
            hook_lora_step = 0
            remove_lora_step = 40
            stable_diffusion.scheduler.ets = []
            stable_diffusion.scheduler.counter = 0
            for i, t in enumerate(stable_diffusion.scheduler.timesteps[-t_ddim:]):
                print(f'reversing timestep {t} as {i} loop')
                if i == hook_lora_step:
                    print('---------------------lora hooked-------------------')
                    stable_diffusion = get_peft_model(stable_diffusion)
                    load_lora_weight(stable_diffusion, lora_path)
                if i == remove_lora_step:
                    print('---------------------lora disabled-----------------')
                    disable_lora(stable_diffusion)
                noise_pred = diffusion_utils.predict(stable_diffusion.unet, noise_latent, t, embeddings)
                latent_pred = diffusion_utils.step(stable_diffusion.scheduler, noise_latent, t, noise_pred)
                noise_latent = latent_pred

            img = diffusion_utils.decode_latent(stable_diffusion.vae, noise_latent)[0]
            attn_scores = stable_diffusion.attn_scores

            path_lora_img = os.path.join(img_folder, 'test_lora.png')
            path_lora_attnmap = os.path.join(img_folder, 'test_lora_attnmap.png')
            path_lora_mergemap = os.path.join(img_folder, 'test_lora_merged_attnmap.png')
            visualize_utils.visualize_merged_attn_score(attn_scores, tokens=tokens, save_path=path_lora_mergemap)
            visualize_utils.visualize_image(img, save_path=path_lora_img)
            visualize_utils.visualize_attn_score(attn_scores[6], tokens=tokens, save_path=path_lora_attnmap)


if score_clip_test:
    path = DEMO_PATH
    images = [diffusion_utils.read_image(path, 'Image')]
    texts = ['a person riding motorbike, a women pushing bicycle in countryside', 'a man in countryside', 'a man riding in countryside on a dirt road', 'riding in coutryside on a dirt road with mountains as background', 'a cat and a dog', 'nothing', '']
    clip = CLIP()
    sim = clip(texts, images)
    print(sim)


if score_regular_test:
    with torch.no_grad():
        path = DEMO_PATH
        lora_path = LORA_SFT_PATH
        folder = TEST_FOLDER_PATH
        save_path1 = os.path.join(folder, 'rl_score.png')
        save_path2 = os.path.join(folder, 'rl_merge_score.png')
        prompt = 'photo of person riding motorbike'
        stable_diffusion = StableDiffusion()
        stable_diffusion = get_peft_model(stable_diffusion)
        load_lora_weight(stable_diffusion, lora_path)
        t_ddim = 8
        num_tokens = diffusion_utils.cnt_tokens(stable_diffusion.tokenizer, prompt)
        tokens = diffusion_utils.get_tokens(stable_diffusion.tokenizer, prompt)

        img, attn_scores = diffusion_utils.pipeline(stable_diffusion, path, t_ddim, prompt)
        visualize_utils.visualize_attn_score(attn_scores[6], tokens, save_path=save_path1)

        # (batch_size, num_tokens, h, w)
        layer_weight=[
            0.1, 0,    # 64*64 downblock
            0.1, 0,    # 32*32 downblock
            0.15, 0.05,    # 16*16 downblock
            0.1, 0.2, 0.3,     # 16*16 upblock
            0.15, 0.2, 0.25,     # 32*32 upblock
            0.1, 0.2, 0.25,     # 64*64 upblock
            0.3     # 8*8 midblock
        ]
        layer_weight=LAYER_WEIGHT
        token_index = 3
        merged_attn_scores = [torch.zeros((64, 64), dtype=torch.float32, device='cuda') for _ in range(num_tokens)]  # supposed to be list of tensor shaped as (77, 64, 64)
        for layer_index, layer_attn_score in enumerate(attn_scores):
            # layer_attn_score shaped as (1, 77, 64, 64)
            layer_attn_score = layer_attn_score[0]
            for i in range(num_tokens):
                token_attn_score = layer_attn_score[i]  # specific token attnmap shaped as (64, 64)
                if token_attn_score.shape != (64, 64):
                    token_attn_score = diffusion_utils.resample(token_attn_score)
                merged_attn_scores[i] += layer_weight[layer_index] * token_attn_score
        merged_attn_scores = torch.stack(merged_attn_scores)

        print(merged_attn_scores.shape)
        merge_score = merged_attn_scores[token_index]
        print(merge_score)

        token_index_disturb = 5
        disturb_score = merged_attn_scores[token_index_disturb]
        
        visualize_utils.visualize_image(visualize_utils.normalize(merge_score), save_path2)

        # critic value
        def softmax(A:torch.Tensor):
            A = torch.nn.functional.softmax(A.view(-1), dim=0).view_as(A)
            return A
        
        def min_max_norm(A:torch.Tensor):
            return (A - A.min()) / (A.max() - A.min() + 1e-8)

        print(softmax(merge_score))

        def entropy_score(A:torch.Tensor)->float:
            alpha = 0.1
            h, w = A.shape
            A = softmax(A)
            entropy = -(A * A.log()).sum()
            max_entropy = math.log(h * w)
            score = (1 - entropy / max_entropy)**alpha
            return score.item()
        
        def overlap_score(A:torch.Tensor, B:torch.Tensor)->float:
            assert A.shape == B.shape
            A = min_max_norm(A)
            B = min_max_norm(B)
            overlap = (A * B).sum() / A.numel()
            score = 1 - overlap
            return score.item()

        def coverage_score(A:torch.Tensor)->float:
            base_precentage = 0.08
            A = softmax(A)
            threshold = A.mean()
            active_ratio = (A > threshold).float().mean()
            score = min((active_ratio / base_precentage).item(), 1.0)
            print(active_ratio)
            return score

        print(entropy_score(merge_score), coverage_score(merge_score), overlap_score(merge_score, disturb_score))
